<a href="https://colab.research.google.com/github/kimjiwoo2/Pill-agent/blob/develop/notebooks/yoonsoo/ys_paddleocr_finetune_v10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pill Imprint OCR – Fine-tuning v10

**v3 → v4 변경 (train/val recognition-only 비교로 확인된 근본 원인 반영)**

진단: pretrained(det+rec)가 val에서 글자정확도 0.811 / EM 0.726인데, v3 fine-tuned(det+rec)는 0.033 / 0.006. recognition-only(detection 없이)로도 fine-tuned가 train(학습에 실제로 쓴 이미지) 91.4%인데 val은 2.8% — train/val은 약 종류가 아예 안 겹치도록 분리되어 있으므로, 이 갭은 **모델이 문자를 읽는 법이 아니라 학습 이미지 전체의 생김새-라벨 매핑을 외웠다(catastrophic forgetting)** 는 뜻. 원인: recognition 학습 데이터를 만들 때 detection 없이 **알약 crop 전체**를 텍스트 한 줄인 것처럼 넣었음 — recognition 모델(PP-OCRv5_server_rec)은 원래 detection이 좁혀준 텍스트 영역만 입력받도록 사전학습된 모델이라, 전혀 다른 입력 분포를 준 셈.

**Fix 5 (이번 핵심)**: Section 9(train)·`setup_local_data`(val 내부 Eval:용) 둘 다, `prepare_img()`로 만든 알약 crop 전체를 그대로 쓰지 않고 **detection(PP-OCRv5_server_det)으로 실제 각인 텍스트 영역을 찾아 그 부분만 크롭**한 뒤 recognition 학습/내부 eval에 사용하도록 변경. eval(Section 12)은 원래부터 det+rec 풀 파이프라인이라 변경 없음 — train 쪽을 eval과 같은 입력 분포로 맞추는 것.
- 1차 시도는 보수적으로: `epoch_num 20→5`, `learning_rate 0.0005→0.00005`, `warmup_epoch 5→1`로 낮춰서 pretrained 가중치를 크게 흔들지 않고 조심스럽게 적응시킴.
- Section 12 평가 직후, **pretrained(det+rec) 기준선과 나란히 비교하는 셀 추가** — fine-tuned가 pretrained(글자정확도 0.811 / EM 0.726 수준)를 못 넘으면 바로 알 수 있도록 함.

**v4 → v5 변경 (v4 코드 리뷰 반영)**
- Fix 6: `crop_text_region`이 여러 박스를 union bbox로 합치던 걸 **박스가 정확히 1개인 경우만 채택**하도록 변경 — 합친 이미지는 CTC 한 줄 모델 기준 불가능한 샘플이라 오히려 암기를 유발할 위험이 있었음. `skip_no_det`/`skip_multi_box`로 원인 분리 집계.
- Fix 7: recognition 학습 데이터를 `rotation_label_quality == 'high'`인 행만 사용하도록 추가 필터 (medium 제외) — medium은 회전각·정답 후보 선택 둘 다 신뢰도가 낮아, 1차 시도는 데이터量보다 순도를 우선.
- Fix 8: `answer_map`의 문자가 `ppocrv5_dict.txt`(CTC 문자셋)에 다 있는지 확인하는 charset 커버리지 체크 셀 추가.
- Section 9 직전에 **최종 학습 입력(prepare_img→detection crop 결과) 눈검증 셀 추가** — 원본 crop만 보던 기존 검증과 달리, 모델이 실제로 받는 이미지가 수평 한 줄 텍스트인지, 회전(crop_v6 기준 라벨을 crop_v7에 적용하는 것이라 프레임이 안 맞을 위험 있음) 문제가 없는지 확인.
- `Backbone.freeze_params: true` 시도는 **제거** — 이 PaddleOCR 버전/백본 조합에서 실제로 지원되는지 검증할 방법이 없어 혼동을 주느니 제외. 낮은 lr(0.00005)·짧은 epoch(5)만 안전장치로 유지.
- Section 12-3의 `text_recognition_model_dir`에 `text_recognition_model_name='PP-OCRv5_server_rec'` 명시 추가, `paddleocr` 버전 고정(`==3.7.0`), Drive 저장 경로 `_v3`→`_v4`. **다만 model_name 유무가 실제 결과에 영향 없다는 것은 이미 라이브 테스트로 확인됨**(0.033/0.006 동일) — recognition-only 테스트가 우리 커스텀 정답 어휘로 train 91.4%를 맞힌 것 자체가 `model_dir`만으로도 올바른 가중치가 로드됐다는 증거이기도 함. 그래도 명시는 위생상 유지.

**v5 → v6 변경 (같은 목적으로 만든 팀원 버전과 교차 검토 후 병합)**
- charset 검사를 리포트만 하던 것에서 **실제 제거**로 변경, 위치도 `answer_map` 로드 직후로 당김 — 이래야 제거된 정답이 coverage_check/filter_answered/Section 3 오버샘플링 통계까지 일관되게 반영됨 (v5는 git clone 이후 뒤늦게 리포트만 하고 실제로 안 지웠음, 이번에 수정).
- Paddle 환경 확인 셀에 `paddleocr.__version__` 출력 추가 (재현성 기록).
- 9-1 최종 입력 눈검증에서 detection 실패 시 빈 칸이 아니라 **원본(회전 보정된) 이미지를 그대로 표시** — 왜 실패했는지 바로 보이도록.
- `crop_text_region`에 `single_only` 토글 추가 (기본 True 유지, False면 기존 병합 방식으로 비교 가능).
- Section 12-3에 `rec_inference_v6` 없으면 `rec_inference_v5`로 자동 폴백 — 재학습 없이 이전 체크포인트를 최신 로딩 코드로 재평가 가능.
- Section 12-5-1을 pretrained + v5 + v6 비교로 확장.

**팀원 버전에서 검토했지만 채택 안 한 것**: `REC_HIGH_ONLY` 필터를 Section 9 루프 안(오버샘플링 이후)에 두는 방식 — 저대비로 3배 뻥튀기된 medium 행이 오버샘플링 통계엔 잡히고 나중에 통째로 버려지는, Fix 4와 같은 종류의 문제가 재발할 수 있어 **기존처럼 Section 3 이전 위치를 유지**하고 토글 기능만 가져옴. `paddleocr<3.7` 캡도 검토했으나, 이번 세션에서 실측 검증된 건 `paddleocr==3.7.0` + `model_name` 명시 조합이라 검증 안 된 버전대로 바꾸는 대신 기존 고정을 유지.

**v6 → v7 변경 (멀티라인 커버리지 복원 + 본격적인 fine-tune 시도)**

v6(=v5, single_only=True로 멀티라인 완전 제외) 결과: 글자정확도 0.816, EM 0.665 — v4(멀티라인 병합 포함, 0.817/0.683)보다 EM이 오히려 하락. 원인: val에는 멀티라인 각인이 여전히 존재하는데, 학습에서 그 유형을 통째로 제외해버려서 모델이 그 카테고리를 아예 못 배움 — 글자 단위 부분점수(글자 정확도)는 완충되지만 전부 맞아야 하는 EM은 직격탄을 맞음. 목표가 "pretrained보다 나은 전체 성능"이므로, 멀티라인 병합의 노이즈(불완전한 학습 샘플)를 감수하고 커버리지를 우선하기로 결정.

- Fix 9: `crop_text_region`의 기본값을 `single_only=False`로 되돌림 (v4와 동일하게 멀티라인도 병합해서 학습에 포함). `single_only=True`는 비교/실험용으로 남겨둠.
- 1차 시도(v4~v6)의 보수적 설정(epoch 5, lr 0.00005)에서 본격적인 fine-tune으로 전환: `epoch_num 5→20`, `learning_rate 0.00005→0.0001`, `warmup_epoch 1→4`, `eval_batch_step [0,50]→[0,150]`. v3(epoch 20, lr 0.0005)의 재앙은 epoch 수 자체가 아니라 입력 분포(Fix 5로 이미 해결)가 원인이었다는 판단 하에, 더 길게/세게 학습해도 되는지 시험.
- Export 체크포인트 선택을 `latest` 우선에서 **`best_accuracy` 우선**으로 되돌림 — v3 때는 `val_rec_label.txt`가 노이즈 라벨이라 `best_accuracy`를 못 믿었지만, v4부터는 `val_rec_label_v3.txt` 기반 내부 eval이 실측으로 신뢰도가 확인됨(epoch별로 꾸준히 상승하는 정상 곡선). 20 epoch까지 늘리면 중간에 정점을 찍고 내려갈 수 있어 정점 epoch을 정확히 골라야 함.
- Section 12-5-1 비교를 pretrained + v6 + v5 + v4까지 연쇄 비교로 확장.

**v7 → v8 변경 (팀원 YOLO OBB straighten crop 파이프라인 도입)**

배경: 팀원이 YOLO OBB 학습을 마치고 알약을 장축 기준으로 수평 정렬해주는 straighten crop 파이프라인을 만듦(분류기·recognition이 같은 crop을 공유 — 분류기는 mod180 정렬로 충분하고, OCR의 잔여 180도는 `use_textline_orientation`이 처리하는 역할 분담). 이 crop은 elongated(캡슐형) 알약만 장축 정렬하고, 원형/정사각 알약은 애초에 장축이 없어 회전을 걸지 않는다(팀원 문서 확인). v1은 원형 알약 각인이 깨지는 버그가 있어 반드시 v2를 사용해야 함.

- **crop 소스 준비**: 팀원 zip(`Pillot/dataset/shared/20k_upright_crops_v2/v2_train.zip`, `v2_val.zip`)은 전체 알약(20k 규모)을 담고 있어, `train_rec_label_v3.txt`/`val_rec_label_v3.txt`에 있는 object_id만 골라내는 별도 노트북(`ys_upright_crop_prepare.ipynb`)을 만들어 `upright_crop_v2_train.zip`/`upright_crop_v2_val.zip`으로 저장 후 사용. `UPRIGHT_VERSION` 변수 하나만 바꾸면 다음 버전 crop에도 재사용 가능.
- **Fix 11 (crop 파이프라인 재정렬 + 180도 보정)**:
  - Section 1: `CROP_ZIP`/`CROP_DIR`를 `crop_v7_train.zip`/`crops_v7_train`에서 `upright_crop_v2_train.zip`/`upright_crop_v2_train`으로 교체. `MANIFEST_CSV`는 print_front/back·split·contrast 정보가 crop 버전과 무관해서 `crop_v7_manifest.csv` 그대로 유지.
  - Section 3-1(회전각 Pseudo Label 로드, `rot_map`)과 Section 7의 `rotation_label_quality` 기반 high-only 필터를 **완전히 삭제** — 둘 다 옛 브루트포스 각도 탐색 라벨링 체계의 산물이라 OBB straighten crop에는 적용 대상 자체가 없음.
  - Section 5 `prepare_img()`: `rotate_image(rot_angle)`·`align_to_long_axis()` 호출 삭제, `load_and_preprocess(CLAHE+Unsharp) → upscale_if_small`만 남김 — straighten crop이 이미 정렬해서 준 입력이므로 CLAHE+Unsharp를 그 뒤(순서 요청대로 straighten 먼저 → 대비/선명도 보정 나중)에 적용하는 셈. `upscale_if_small`도 매니페스트의 `bbox_w/h`(옛 crop 스케일 기준) 대신 실제 로드된 이미지 크기로 판단하도록 변경 — crop 버전이 바뀌어도 안전.
  - Section 6-1: `TextLineOrientationClassification`(`PP-LCNet_x1_0_textline_ori`)을 새로 로드해 `resolve_180_flip()` 함수 추가 — detection crop 직후 텍스트가 180도 뒤집혔으면 바로잡음. Section 9(학습 데이터 생성)와 `setup_local_data`(val 재생성)에서 `crop_text_region()` 직후 호출. **주의: 클래스명/응답 키(`label_names` 등)는 라이브 검증 전 추정값 — 로드 실패해도 조용히 180도 보정만 건너뛰고 나머지는 정상 진행되도록 try/except로 방어했으니, 실행 후 프린트되는 실제 키/값을 보고 다르면 그 셀만 수정할 것.**
  - **Fix 12 (v8 학습 후 실측 결과 반영 — v2 crop 도입 직후 지표 급락 원인 분석에서 도출)**: `crop_text_region`이 지금까지 poly의 min/max로 축 정렬된 bbox만 자르고 회전은 안 했다는 걸 재확인. 원형/정사각 알약은 OBB straighten 단계에서 회전을 아예 안 걸어서 임의 각도(0~360°)로 남는데, 이 무보정 텍스트를 그대로 학습 라벨과 짝지어 넣고 있었던 게 이번 v8 첫 결과(글자정확도 0.559/EM 0.408, pretrained도 0.545/0.420으로 동반 하락)의 핵심 원인으로 추정됨. `_long_axis_angle_deg`를 추가해 **알약이 아니라 감지된 텍스트 poly 자체의 장축각**으로 ±90도 회전 보정 후 크롭하도록 `crop_text_region`을 수정 — 텍스트는 알약 모양과 무관하게 항상 길쭉한 한 줄이라 원형 알약도 커버됨. 멀티박스(멀티라인)는 poly별 각도를 면적 가중 원형평균으로 묶어 대표각 산출.
- **Fix 13 (train/eval 정렬 로직 통일)**: Fix 12를 처음엔 Section 9/`setup_local_data`(train 쪽)에만 넣고, Section 12(최종 eval)는 `PaddleOCR()` 풀 파이프라인이 내부적으로 유사한 회전 보정을 이미 하고 있을 거라 가정하고 손대지 않았음. 하지만 이 가정이 검증 안 된 채로 train/eval이 서로 다른 코드로 정렬하면 그 자체가 새로운 학습-평가 입력 불일치 리스크가 된다는 지적을 받아, Section 12의 `run_inference_val`도 recognition 호출 직전에 동일하게 `crop_text_region`(Fix 12) + `resolve_180_flip`(Fix 11)을 거치도록 변경. 이제 train·eval 세 경로(Section 9, `setup_local_data`, Section 12) 전부 물리적으로 같은 정렬 코드를 탄다. 부수적으로, eval에서 우리 detection이 텍스트 영역을 못 찾는 경우(정답 후보는 있는데) 이전엔 글자정확도 계산에서 조용히 제외됐는데, 이제는 명시적으로 0점(완전 오답) 처리해서 통계가 survivorship bias로 부풀려지지 않도록 함.
  - Section 12-3/12-5-1(fine-tuned·pretrained 둘 다): `use_textline_orientation=False`→`True`로 변경 — eval에서는 PaddleOCR 풀 파이프라인이 이 옵션으로 180도를 자체 처리하므로 별도 `resolve_180_flip` 호출 불필요.
  - Section 12-4(val 데이터 로드)의 `VAL_CROP_ZIP`/`VAL_CROP_DIR`도 `crop_v7_val.zip`에서 `upright_crop_v2_val.zip`으로 교체 (놓치기 쉬운 지점 — train만 바꾸고 이쪽을 빠뜨리면 eval이 여전히 옛 crop으로 도는 사고가 남).
- **알려진 한계(v8에서도 미해결)**: Fix 12는 detection이 텍스트를 찾아낸 경우에만 적용된다. detection 자체가 실패하는 케이스(Section 9 로그 `detection 실패로 제외` 건수)는 각도를 잴 poly 자체가 없어 여전히 구제 못 함 — 회전 각도를 바꿔가며 detection을 재시도하는 등의 보완이 다음 과제로 남음.
- Drive 저장 경로 `_v7`→`_v8`(`rec_finetune_v8`, `rec_inference_v8`, `train_rec_label_augmented_v8.txt`), 결과 CSV `ft_v8_val_result.csv`, Section 12-5-1 비교 체인에 `ft_v7_val_result.csv` 추가.

**v8 → v9 변경 (코드 리뷰 반영 — P1 3건, P2 2건)**

v8 학습 후 노트북 전체를 리뷰받아 5개 지적사항을 반영. 우선순위는 최종 성능 해석에 영향을 주는 것부터.

- **Fix 14 [P1] (내부 eval subset vs 최종 eval 전체집합 불일치, + Fix 13의 0점 처리 수정)**: Section 9(train)와 `setup_local_data`(내부 val) 둘 다 detection 실패 샘플을 라벨 파일에서 통째로 제외하고 있었는데, `tools/train.py`의 `best_accuracy`는 이 내부 val로 선택된다. 반면 Fix 13에서 Section 12(최종 eval)는 detection 실패를 0점으로 채점하게 만들어서, best_accuracy는 'detection이 성공한 쉬운 부분집합' 기준으로 뽑히고 최종 성능은 '실패 포함 전체 집합' 기준으로 떨어지는 모집단 불일치가 생겼음(리뷰 지적). Section 9 / `setup_local_data` / Section 12 세 곳 모두, detection 실패 시 제외하거나 강제 0점 처리하는 대신 **전체 crop(`prepare_img` 결과)으로 fallback해서 그대로 recognition을 시도**하도록 통일 — 완벽히 정렬은 안 됐어도 학습·내부 eval·최종 eval이 항상 같은 모집단을 보게 됨. Section 9/`setup_local_data`는 `fallback_whole_crop`(_val) 카운트를, Section 12는 `error` 필드에 fallback 여부를 남겨서 추적 가능.
- **Fix 15 [P1] (Drive stale 모델을 몰래 평가하는 위험)**: Section 12-3에서 Drive의 `rec_inference_v8`이 없으면 조용히 `rec_inference_v7`(구버전)로 폴백하던 구조를 제거. 이제 ①로컬에 방금 export한 `/content/rec_inference`가 이미 있으면 Drive 왕복 없이 그대로 사용, ②로컬에 없으면 Drive의 `rec_inference_v9`만 복원하고 **없으면 예전 버전으로 폴백하지 않고 assert로 명시적으로 멈춤**. 잘못된(예전) 모델을 평가하고도 모르고 넘어가는 사고를 원천 차단.
- **Fix 16 [P1] (PaddleOCR 코드/사전 버전 미고정)**: `paddleocr==3.7.0`(pip, 추론용)과 학습에 쓰는 `git clone .../PaddleOCR`(main 브랜치, 학습 스크립트+dict)이 서로 다른 시점 조합일 수 있다는 지적. 정확히 대응하는 태그를 안전하게 확정할 방법이 없어 clone은 main 그대로 두되, **실제로 어떤 커밋이 쓰였는지 항상 출력**하도록 해서 재현이 필요하면 그 해시로 고정할 수 있게 함. 또한 Section 7(OOV 체크)과 실제 학습(`character_dict_path`)이 dict 파일을 각자 다른 시점에 GitHub main에서 따로 받고 있던 걸, **Section 7이 받은 `/content/ppocrv5_dict.txt` 하나만 공유**하도록 통일 — 세션 중간에 main이 바뀌어도 OOV 필터링과 실제 학습이 항상 같은 dict를 보게 됨.
- **[P2] train 쪽도 detector 의존적이라 hard case가 학습에서 빠지는 문제**: Fix 14와 같은 코드로 이미 해결됨(전체 crop fallback이 train에도 적용됨).
- **[P2] batch_size/drop_last로 매 epoch 버려지는 잔여 샘플**: Section 9 마지막에 `len(train_rec_label) % batch_size`를 출력하는 로그 추가 — 지금 규모(학습 샘플 약 3.5만 개, batch 256)에서는 매 epoch 최대 255개(1% 미만) 손실이라 심각하진 않지만, 데이터 규모가 바뀌면 이 숫자로 바로 확인 가능.

**v9 내 추가 수정 — Fix 17 (9-2 진단 셀 결과 반영)**

9-2 진단 셀을 돌려본 결과: detection 실패 300건 샘플 중 39건이 실패, 그중 56.4%(22건)가 **회전만 시키면 detection 성공**으로 확인됨(면적은 성공/실패 그룹 간 차이 없음, blur_score는 실패 그룹이 다소 낮아 흐림이 부분적 원인으로 추정). 회전 문제가 과반을 차지한다는 게 확인돼서 `crop_text_region`에 재시도 로직 추가: **최초 detection이 박스 0개로 완전히 실패한 경우에만** 90/180/270도로 돌려 재시도(모든 crop에 일괄 적용 아님 — 정상 케이스는 재시도 비용 없음). 재시도로도 못 찾는 나머지는 그대로 Fix 14의 전체 crop fallback으로 학습/eval에 포함됨.

**v9 내 추가 수정 — Fix 18 (crop_v7로 원복)**

v8/v9에서 시도한 YOLO OBB straighten crop(upright_crop_v2)을 그만두고 crop_v7 + `rotation_label_deg`(옛 브루트포스 각도 탐색 라벨) 방식으로 되돌림. 근거: pretrained 모델(파인튜닝 전혀 안 받음)까지 upright_crop_v2에서 0.545/0.420으로 crop_v7 기준(0.829/0.737) 대비 같이 무너져서, 문제가 학습 방식이 아니라 OBB crop 자체(특히 원형/정사각 알약이 회전 보정을 아예 못 받는 설계)에 있다고 판단.

- Section 1: `CROP_ZIP`/`CROP_DIR`를 `crop_v7_train.zip`/`crops_v7_train`으로 원복.
- Section 3-1(`rot_map`)과 Section 7(`REC_HIGH_ONLY` 필터)을 재도입. **`REC_HIGH_ONLY=False`로 설정**(high+medium 둘 다 사용) — medium 라벨의 낮은 신뢰도를 Fix 12(poly 각도 재측정)·Fix 17(재시도)·Fix 11(180도 재확인)이 어느 정도 보완해줄 수 있다고 판단해서, 데이터 양을 우선.
- Section 5: `prepare_img(crop_path, bbox_w, bbox_h, rot_angle=None)` 시그니처와 `align_to_long_axis`를 복원. `rotate_image(-rot_angle)` → `align_to_long_axis`(90도 단위 마무리) → `upscale_if_small(bbox_w, bbox_h)` 순서로 v7과 동일하게 동작.
- **Fix 12/17/11(OBB 대응으로 만든 poly 각도 보정·재시도·180도 확인)은 전부 유지**. `rotation_label_deg`는 추정값이라 잔차 오차가 남을 수 있어서, 이 셋이 '라벨이 대략 맞춰놓은 것을 실측으로 미세 보정'하는 보완 역할로 여전히 유효함(OBB 때는 원형 알약의 전체 회전 문제를 보완했다면, 지금은 rotation_label_deg의 잔차 오차를 보완).
- Section 9 / `setup_local_data` / Section 12-1: `bbox_w`/`bbox_h`/`rot_map`(_val) 조회해서 `prepare_img`에 넘기는 코드 복원. Fix 14(전체 crop fallback)/Fix 13(train·eval 정렬 로직 통일)은 그대로 유지.
- Section 12-3: `use_textline_orientation`은 v7의 `False`로 되돌리지 않고 **`True` 유지** — `run_inference_val`이 이미 `crop_text_region`+`resolve_180_flip`을 거친 뒤라 대부분 무해한 안전망으로만 작동하고, `rotation_label_deg`의 방향 판단이 틀렸을 경우의 이중 안전장치가 됨.
- Section 12-4: `VAL_CROP_ZIP`을 `crop_v7_val.zip`으로 원복.

**v9 내 추가 수정 — Fix 19 (rotation_label_deg data leakage 발견 및 블라인드 eval 추가)**

`rotation_label_deg`가 **정답 텍스트와 대조해서** 고른 라벨이라는 게 확인됨(브루트포스로 여러 각도 OCR 결과 중 정답과 일치하는 각도 채택) — 명백한 data leakage. Section 9/`setup_local_data`(학습데이터 생성)에 쓰는 건 문제 없음(정답을 참고해 깨끗한 학습셋을 만드는 정상적 데이터 큐레이션이라 그대로 둠). 하지만 **Section 12(최종 eval)는 '실제 배포 시 성능'을 재려는 지표라서 이 라벨을 쓰면 안 됨** — 실제 배포에서는 새 알약 사진의 정답을 모르니 이런 각도 힌트를 못 받는다.

**Section 12-5-2 추가**: `rot_angle=None`으로 돌려서 `align_to_long_axis`(휴리스틱) + `crop_text_region`(Fix 12/17, 실측 기반) + `resolve_180_flip`(Fix 11)만으로 회전을 처리하는 `run_inference_val_blind()`를 만들어, 기존 '라벨 사용' 결과와 나란히 비교 출력. 이 격차가 곧 라벨 leakage가 지금까지의 보고 성능을 얼마나 낙관적으로 부풀렸는지를 보여준다.

**주의**: `tools/train.py`가 `best_accuracy` 선택에 쓰는 내부 val(`setup_local_data`가 만든 `rec_val`)은 여전히 라벨을 사용한다 — 이건 학습 중 체크포인트 선택용 내부 신호라 Section 9와 같은 논리로 그대로 둠. 실제 배포 성능의 정직한 추정치는 Section 12-5-2(블라인드) 쪽만 봐야 함.

**v9 내 추가 수정 — Fix 19 (Section 12 eval의 회전 라벨 leakage 제거)**

`rotation_label_deg`가 실은 **정답 텍스트와 대조해서 각도를 고른 라벨**이라는 걸 확인함 — 실제 배포에서는 정답을 모르니 이 라벨 자체가 존재할 수 없는데, Section 12(eval)에서 이걸 써서 각도를 맞춰준 뒤 채점하면 실제 배포 성능보다 부풀려진 수치가 나옴(data leakage). YOLO OBB의 `pred_angle_deg`(`angle_reliable`)도 대안으로 검토했으나 신뢰 가능한 각도가 전체의 7%뿐이라 기각.

대신 **OCR 자신의 recognition confidence를 각도 선택 기준으로 사용** — conf-CER 상관 -0.734로, confidence가 높을수록 실제로 잘 읽었을 가능성이 높다는 게 확인됨. 정답도 YOLO도 필요 없이 이미지 한 장만으로 계산 가능해서 실제 배포에서 그대로 재현 가능한 방식.

- `resolve_rotation_by_confidence(img, ocr, angle_step=30)` 함수 추가(Section 12-1): 0~330도를 30도 간격(12번)으로 돌려보고, 각 각도마다 `crop_text_region`(Fix 12)+`resolve_180_flip`(Fix 11)을 거친 뒤 recognition confidence를 비교해서 가장 높은 각도의 결과를 채택.
- `run_inference_val`이 `rot_map_val`(`rotation_label_deg` 기반) 대신 이 함수를 사용하도록 변경. 12개 각도 전부 실패해도 전체 crop으로 마지막 시도(Fix 14 정신 유지 — 완전 포기 안 함).
- **Section 9(학습데이터 생성)는 `rotation_label_deg` 그대로 유지** — 정답을 참고해 좋은 학습 페어를 만드는 것 자체는 leakage가 아니라 정상적인 데이터 큐레이션이라 문제없음. leakage는 '실제 배포에서 못 구할 정보를 성능 측정에 쓰는 것'이 핵심이라 eval(Section 12)에서만 문제.

**주의(다음 과제로 남김)**: `setup_local_data`(학습 중 내부 eval, `best_accuracy` 선택 기준)는 아직 `rotation_label_deg` 기반(`rot_map_val_tmp`)을 그대로 씀. 그래서 지금 `best_accuracy`는 'leakage로 완벽 정렬된 이미지 기준'으로 뽑히는데, Section 12 최종 성능은 'confidence 탐색 기준'으로 측정돼서, Fix 14로 한번 맞췄던 모집단 일치가 **입력 처리 방식** 차원에서 다시 어긋났다. 완전히 일치시키려면 `setup_local_data`도 confidence 탐색으로 바꿔야 하는데, 이러면 내부 eval마다(매 150 iteration) val 세트 전체를 12배로 다시 도는 비용이 들어서 학습 시간이 크게 늘어날 수 있음 — 일단 보류.

**v9 내 추가 수정 — Fix 20 (multi-angle 탐색, 정확성 우선으로 최종 확정)**

효율화를 위해 `resolve_180_flip`을 1등 각도 뽑은 뒤 한 번만 적용하는 방식을 시도했었으나(모델 호출 36번 → 약 25~26번), **진짜 정답 각도가 마침 180도 뒤집힌 상태로 후보에 들어오면 뒤집힌 채로 측정된 confidence가 낮게 나와 다른 오답 각도한테 밀려 탈락할 위험**이 있어 정확성을 위해 되돌림. 최종적으로 `resolve_rotation_by_confidence`는 **12개 후보 각도 전부에 `crop_text_region`+`resolve_180_flip`을 다 적용한 뒤** confidence를 비교하는 방식 유지 — 모든 후보를 '뒤집기까지 적용한 뒤의 진짜 잠재력'으로 공정하게 비교해야 최적 각도를 놓치지 않음. Section 12는 val 세트에만 도는 eval이라 계산량 차이(약 36번 vs 25~26번)가 감당 못 할 수준은 아니라고 판단.

**v9 내 추가 수정 — Fix 21 (학습 중 Drive 자동 백업)**

런타임 연결이 끊겨 새 VM이 배정되면 `/content`(휘발성)의 체크포인트가 전부 사라져서 처음부터 다시 돌려야 하는 사고 발생(epoch 9/20까지 진행된 상태에서 소실). 재발 방지로 학습 셀을 `!python tools/train.py` 셸 실행 대신 `subprocess.Popen`으로 백그라운드 실행하면서, **10분마다 `output/rec_finetune`을 Drive(`rec_finetune_v9_inprogress`)에 자동 백업**하도록 변경. 끊겨도 최대 10분치 진행만 손실되고, `Global.checkpoints=<복원 경로>/latest`로 이어서 재개 가능.

**v9 내 추가 수정 — Fix 22 (Section 12 eval: crop_text_region → rotate_only 교체)**

pretrained 모델에도 v9 전처리(crop_text_region+resolve_180_flip)를 이식해서 성능이 오르는지 확인하려고 별도 ablation 노트북들(`ys_pretrained_ocr_score_v7_preprocessing_ablation.ipynb`, `_4way.ipynb`, `_v2.ipynb`)로 실측한 결과, 오히려 성능이 떨어졌다(baseline 0.826/0.718 → crop 적용 후 0.750/0.639 EM). 4-way/추가 ablation으로 원인을 분해해보니 회전 보정 자체는 오히려 소폭 도움이 됐고(+0.007~+0.019), **`crop_text_region`의 타이트 크롭이 손실의 거의 전부**였다(패딩을 0.12→0.4로 넓혀도 별 차이 없음, 여전히 baseline보다 낮음). 원인: Section 12는 자체 detection이 내장된 풀 `PaddleOCR()` 파이프라인인데, 우리가 미리 타이트하게 잘라서 넘기면 그 내장 detection이 스스로 잘 찾을 수 있었던 문맥을 뺏는 이중 손실이 된다 — 반면 Section 9(recognition 전용 모델 학습 데이터 생성)는 내장 detection이 없는 recognition-only 학습이라 미리 잘라주는 게 여전히 필수다.

그래서 `crop_text_region`과 각도 탐색 로직은 동일하되 마지막 크롭 단계만 없는 `rotate_only()` 함수를 추가하고, Section 12-1 `resolve_rotation_by_confidence`가 각 후보 각도마다 `crop_text_region` 대신 `rotate_only`를 쓰도록 교체(회전 보정 + `resolve_180_flip`은 유지, 크롭만 제거). Section 9와 `setup_local_data`는 그대로 `crop_text_region` 유지.

**v9 내 추가 수정 — Fix 23 (Fix 22 재검증: rotate_only 제거, 순수 multi-angle만 유지)**

Fix 22에서 Section 12 eval의 `crop_text_region`을 `rotate_only`(크롭 없이 회전만 보정)로 교체했는데, 후속 ablation(`ys_pretrained_ocr_score_v7_preprocessing_ablation_v2.ipynb`)에서 크롭을 완전히 뺀 상태로 `rotate_only` 단독을 적용해도 baseline(아무 후처리 없이 원본을 풀 파이프라인에 그대로 넘김)보다 더 나쁘게 나왔다(0.826/0.718 → 0.772/0.662, EM -0.056). 즉 손실의 원인이 크롭만이 아니라 poly 각도 기반 미세 회전보정(`rotate_only`) 자체에도 있었다는 뜻 — conf-CER 상관이 -0.734로 완벽하지 않다 보니, 이미 충분히 잘 읽히는 이미지에 불필요한 재보정(및 재보간)을 걸면 오히려 텍스트 품질을 떨어뜨리는 쪽으로 작용한 것으로 보인다. **주의**: 이 arm은 `resolve_180_flip`을 호출하지 않았다 — baseline을 포함한 세 arm 모두 180도 처리는 `use_textline_orientation=True` 내장 로직에만 의존했다. 즉 `resolve_180_flip`이 나쁜 영향을 줬다는 근거는 이 실험에 없고(3-way/4-way ablation에서 크롭 위에 얹었을 때는 오히려 근소하게 긍정적이었음, +0.001~+0.003), '독'으로 실측 확인된 건 `rotate_only`뿐이다.

반면 30도 단위로 이미지를 통째로 돌려보고(`rotate_image`) confidence로 최적 각도를 고르는 multi-angle 탐색 자체는 (하고 안 하고 비교에서) 유의미하게 도움이 되는 것으로 확인됨 — 알약 사진은 실제로 0~360도 임의 방향으로 촬영되기 때문에, 큰 방향 오차를 굵은 단위로 바로잡는 건 여전히 필요하다.

그래서 `resolve_rotation_by_confidence`에서 `rotate_only(rimg)` 호출을 제거하고, `rotate_image(img, angle)`로 30도 단위 회전만 적용한 이미지를 그대로 OCR에 넣어 confidence로 최적 각도를 고르도록 단순화했다. `resolve_180_flip`도 같이 뺐는데, 이는 '나쁘다고 확인돼서'가 아니라 근거 없이 12번 반복 호출을 추가할 이유가 없어서다(효과가 있어도 근소함). 더 이상 쓰이지 않는 `rotate_only()` 함수 정의 자체도 Section 6-1과 pretrained 노트북 양쪽에서 삭제. Section 9/`setup_local_data`는 영향 없음(`crop_text_region` 그대로 유지).

**v9 → v10 변경 (팀 결정: val도 학습 데이터에 포함)**

**Fix 24**: 팀 내부에서 라벨 데이터가 부족하니 val split도 학습에 포함시키기로 결정함. 지금까지 Section 12(최종 eval)와 Section 11 내부 checkpoint 선택(`best_accuracy`) 모두 `split == 'val'` 데이터를 학습에서 완전히 제외한 채로 held-out 평가에만 써왔는데, 이제 이 격리를 포기한다.

- **Section 9-3 (신규)**: `crop_v7_val.zip` + `val_rec_label_v3.txt`를 train과 동일한 파이프라인(`prepare_img` → `crop_text_region` → `resolve_180_flip` → `augment_image`)으로 처리해서 `train_rec_label.txt`에 이어붙인다. Section 9(train split)의 기존 로직은 그대로 두고, 그 뒤에 val split도 추가로 편입하는 방식 — train 쪽 변경은 없음.
- **Section 11(`setup_local_data`)은 코드 변경 없음**: 여전히 `val_rec_label_v3.txt` 기준으로 `rec_val`(Eval 데이터셋)을 만들지만, 그 이미지들이 이제 Train에도 들어있어서 학습 중 `best_accuracy` 선택 기준이 더 이상 순수한 held-out 지표가 아니게 됨(부분적으로 암기된 데이터로 채점). 별도 신규 holdout을 새로 떼어내는 대신 이 leakage를 그대로 감수하기로 함(팀 결정).
- **Section 12(최종 eval)도 코드 변경 없음**: 여전히 `val_rec_label_v3.txt`로 채점하지만 이제 그 이미지들이 학습에 쓰였으므로 결과가 leakage로 부풀려짐 — **v10의 Section 12 숫자는 v9까지의 "honest" 숫자와 직접 비교할 수 없다.** 상대적인 참고용으로만 사용할 것.
- Drive 저장 경로 `_v9`→`_v10`(`rec_finetune_v10`, `rec_inference_v10`, `train_rec_label_augmented_v10.txt`, `rec_finetune_v10_inprogress`), 결과 CSV `ft_v10_val_result.csv`/`ft_v10_blind_val_result.csv`, Section 12-5-1 비교 체인에 `ft_v9_val_result.csv` 추가.


## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip uninstall -y torch torchvision torchaudio modelscope
!pip install paddlepaddle-gpu==3.1.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/
!pip install 'paddleocr==3.7.0'  # 버전 고정: 3.7.0에서 검증됨
!pip install -q opencv-python-headless pandas numpy matplotlib tqdm

In [ ]:
import matplotlib, matplotlib.font_manager as fm
!apt-get install -y fonts-nanum > /dev/null 2>&1
fm.fontManager.__init__()
matplotlib.rc('font', family='NanumGothic')
matplotlib.rcParams['axes.unicode_minus'] = False

In [ ]:
from __future__ import annotations
import json, re, zipfile, shutil, os
from dataclasses import dataclass
from pathlib import Path
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

## 1. 경로 설정

In [ ]:
DRIVE_BASE    = Path('/content/drive/MyDrive/Pillot/yoonsoo')
CROP_ZIP      = DRIVE_BASE / 'crop_v7_train.zip'  # Fix 18: OBB(v8/v9) 실험 종료, crop_v7 + rotation_label_deg로 원복
MANIFEST_CSV  = DRIVE_BASE / 'crop_v7_manifest.csv'  # 회전각(rotation_label_deg) + bbox_w/h 포함된 매니페스트
CROP_DIR      = Path('/content/crops_v7_train')
DET_TRAIN_DIR = Path('/content/det_train')
REC_TRAIN_DIR = Path('/content/rec_train')
SCRATCH_CROPS = Path('/mnt/local-scratch/crops')  # ~350GB 스크래치 디스크
RESULT_DIR    = DRIVE_BASE / 'results'
for d in [CROP_DIR, DET_TRAIN_DIR, REC_TRAIN_DIR, RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print(f'CROP_ZIP: {CROP_ZIP}  exists={CROP_ZIP.exists()}')
print(f'MANIFEST: {MANIFEST_CSV}  exists={MANIFEST_CSV.exists()}')

In [ ]:
import zipfile
from pathlib import Path
from tqdm.auto import tqdm

existing = {p.name for p in CROP_DIR.iterdir()}
print(f'이미 있는 파일: {len(existing)}개')

with zipfile.ZipFile(CROP_ZIP, 'r') as zf:
    members = zf.infolist()
    missing = [m for m in members if Path(m.filename).name not in existing]
    print(f'압축 내 전체: {len(members)}개  |  미해제: {len(missing)}개')

    for m in tqdm(missing, desc='재해제'):
        zf.extract(m, CROP_DIR)

print('완료')

## 2. Manifest 로드 & target 정규화

In [ ]:
IGNORE_IMPRINT_TOKENS = {'', 'NAN', 'NONE', 'NULL', '마크', '분할선', '없음', '무', '-', '십자'}
_RE_STRIP_TOKENS = re.compile(r'\s+|분할선|마크|\|')
_RE_ALLOWED      = re.compile(r'[^0-9A-Z가-힣+\-/]')

def normalize_imprint(text):
    if pd.isna(text): return ''
    text = str(text).strip().upper()
    if text in IGNORE_IMPRINT_TOKENS: return ''
    text = _RE_STRIP_TOKENS.sub('', text)
    text = _RE_ALLOWED.sub('', text)
    return '' if text in IGNORE_IMPRINT_TOKENS else text

def build_target_for_row(row):
    front = normalize_imprint(row.get('print_front', ''))
    back  = normalize_imprint(row.get('print_back',  ''))
    cands = list(dict.fromkeys(t for t in [front, back] if t))
    return {'target_text_front': front, 'target_text_back': back,
            'target_candidates': cands, 'target_text': '/'.join(cands)}

In [ ]:
df_manifest = pd.read_csv(MANIFEST_CSV, low_memory=False)
df_manifest = df_manifest[df_manifest['split'] == 'train'].reset_index(drop=True)
print(f'manifest: {df_manifest.shape}')
df_manifest['crop_path'] = df_manifest['object_id'].apply(
    lambda oid: str(CROP_DIR / f'{oid}.png'))
tgt = df_manifest.apply(build_target_for_row, axis=1, result_type='expand')
df_manifest = pd.concat([df_manifest, tgt], axis=1)
df_manifest['_exists'] = df_manifest['crop_path'].apply(lambda p: Path(p).exists())
df = df_manifest[df_manifest['_exists']].reset_index(drop=True)
print(f'사용 가능: {len(df)}건')
display(df[['object_id','target_text_front','target_text_back','target_text']].head())

## 2-1. 정답 매핑 로드 (`train_rec_label_v3.txt`) + 눈으로 검증

**Fix 1**: 예전 버전은 여기서 pseudo_labels_v6_full.csv(OCR 결과) → `score_one` 기반 cutoff → 후보 선택 로직으로 정답을 매번 다시 만들었음. 그런데 이 cutoff가 `score_one`의 substring 버그를 그대로 물려받아 오염된 라벨이 섞여 있었고, 증강 후 파일명(`{object_id}_{idx}_{face}.png`)으로 Drive에 백업되면서 원본 `crop_v6_train.zip`(`{object_id}.png`)과 형식이 어긋나 눈으로 검증할 방법이 없었음.

**v3**: 다른 노트북에서 양면 각인까지 이미 필터링해 object_id당 정답 하나만 남긴 `DRIVE_BASE/train_rec_label_v3.txt`를 그대로 로드. 재생성하지 않음 — "어떤 사진을 학습에 쓸지 / 그 사진의 정답이 뭔지"를 매핑한 소스 오브 트루스로 취급.

In [ ]:
answer_lines = (DRIVE_BASE / 'train_rec_label_v3.txt').read_text(encoding='utf-8').strip().split('\n')
answer_map = {}
dup_mismatch = 0
for line in answer_lines:
    parts = line.split('\t')
    if len(parts) != 2:
        continue
    oid  = Path(parts[0]).stem  # crops/{object_id}.png → object_id
    text = parts[1].strip()
    if oid in answer_map and answer_map[oid] != text:
        dup_mismatch += 1
    answer_map[oid] = text

print(f'train_rec_label_v3.txt 로드: {len(answer_lines)}줄 → object_id {len(answer_map)}개')
print(f'중복 object_id 중 정답 불일치: {dup_mismatch}건')
display(pd.DataFrame(list(answer_map.items())[:10], columns=['object_id', 'answer']))

In [ ]:
# 사전(charset) 검사: 정답에 ppocrv5_dict에 없는 문자가 있으면 CTC 타깃이 조용히 오염됨 -> 미리 제거.
# 여기서 지워야 아래 coverage_check(_has_answer)/filter_answered/Section 3 오버샘플링 통계까지
# 전부 '실제로 학습에 쓰일 정답' 기준으로 일관되게 계산됨 (git clone 이후로 미루면 이미 늦음).
import urllib.request
_dict_path = Path('/content/ppocrv5_dict.txt')
if not _dict_path.exists():
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PaddlePaddle/PaddleOCR/main/ppocr/utils/dict/ppocrv5_dict.txt',
        str(_dict_path))
charset = set(_dict_path.read_text(encoding='utf-8').split('\n'))
oov = {oid: t for oid, t in answer_map.items() if any(ch not in charset for ch in t)}
if oov:
    print(f'[charset] 사전에 없는 문자가 든 정답 {len(oov)}건 제거 (예: {list(oov.items())[:5]})')
    for oid in oov:
        del answer_map[oid]
else:
    print('[charset] 모든 정답 문자가 사전에 포함됨')
print(f'charset 검사 후 answer_map: {len(answer_map)}개')

In [ ]:
# manifest 커버리지 + front/back 후보와의 정합성 체크
df['_has_answer'] = df['object_id'].isin(answer_map)
print(f'manifest 대비 정답 매핑 커버리지: {df["_has_answer"].sum()}/{len(df)}건 '
      f'(누락 {(~df["_has_answer"]).sum()}건 — 이 건들은 Section 9에서 학습 데이터로 사용되지 않음)')

def _norm_eq(a, b):
    return normalize_imprint(a) == normalize_imprint(b)

mismatch_rows = []
for _, row in df[df['_has_answer']].iterrows():
    ans   = answer_map[row['object_id']]
    cands = row['target_candidates']
    if cands and not any(_norm_eq(ans, c) for c in cands):
        mismatch_rows.append((row['object_id'], ans, cands))

print(f'정답이 manifest 후보(front/back)와 전혀 안 맞는 행: {len(mismatch_rows)}건')
if mismatch_rows:
    display(pd.DataFrame(mismatch_rows[:20], columns=['object_id', 'answer', 'candidates']))

### 2-1-1. 눈으로 검증

`train_rec_label_v3.txt`의 정답과 `crop_v7_train.zip`(Fix 18: crop_v7 원복)에서 풀린 실제 이미지(`CROP_DIR/{object_id}.png`)를 랜덤 샘플로 나란히 띄워 육안 대조. **여기서 이상하면 다음 셀로 넘어가지 말고 정답 매핑을 만든 노트북부터 다시 확인할 것.**

In [ ]:
sample_ids = list(answer_map.keys())
rng_check  = np.random.default_rng(0)
sample_ids = list(rng_check.choice(sample_ids, size=min(16, len(sample_ids)), replace=False))

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for ax, oid in zip(axes.flat, sample_ids):
    img_path = CROP_DIR / f'{oid}.png'
    if img_path.exists():
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f'{oid}\nGT: {answer_map[oid]}', fontsize=9)
    else:
        ax.set_title(f'{oid}\n[파일 없음]', fontsize=9, color='red')
    ax.axis('off')
plt.tight_layout()
plt.show()
print('▲ 16장의 정답 텍스트가 실제 이미지 각인과 일치하는지 확인. 매번 다른 샘플을 보려면 seed를 바꿔 재실행.')

In [ ]:
# Fix 4: 정답 매핑 없는 행은 여기서 미리 제거.
# → Section 3의 대비 분석/오버샘플링 통계가 '실제로 학습에 쓰일 행' 기준으로 계산되도록 함.
# (기존에는 오버샘플링을 먼저 하고 정답 필터링을 Section 9에서 뒤늦게 적용해서,
#  저대비로 3배 뻥튀기된 행이 정답이 없으면 3장 모두 통째로 버려지는데도 Section 3 통계에는 안 잡혔음)
before_n = len(df)
df = df[df['_has_answer']].reset_index(drop=True)
print(f'정답 매핑 있는 행만 유지: {before_n}건 → {len(df)}건 (제외 {before_n - len(df)}건)')

In [ ]:
# Fix 7: rotation_label_quality가 high인 행만 사용할지 토글 (medium 제외).
# medium = 회전각 산출 과정에서 OCR이 어떤 각도에서도 완전히 확신하지 못한 라벨이라,
# 회전각·정답 후보 선택 둘 다 신뢰도가 낮음.
# Fix 18: v9에서는 crop_text_region(Fix 12, poly 각도로 재측정) + resolve_180_flip(Fix 11) +
# 재시도(Fix 17)가 rotation_label_deg의 잔차 오차를 어느 정도 보완해줄 수 있다고 판단해서,
# medium도 포함(REC_HIGH_ONLY=False)해서 데이터 양을 우선함.
# 반드시 Section 3(오버샘플링) 이전에 걸러야 함 — 오버샘플링 이후에 거르면 저대비로 3배
# 뻥튀기된 medium 행이 통째로 버려지는데도 Section 3 통계에는 안 잡히는 문제가 생김(Fix 4와 동일 원리).
REC_HIGH_ONLY = False  # True로 바꾸면 high만 사용 (medium 제외)
if REC_HIGH_ONLY:
    before_n = len(df)
    df = df[df['rotation_label_quality'] == 'high'].reset_index(drop=True)
    print(f'REC_HIGH_ONLY: {before_n}건 → {len(df)}건 (medium 제외)')
else:
    print(f'REC_HIGH_ONLY=False: high+medium 전체 {len(df)}건 사용')

## 3. 대비 분석 & Oversampling

- contrast ≤20 (저대비): x3 oversampling
- contrast 20~40: 기준 품질

In [ ]:
df_contrast = pd.read_csv(DRIVE_BASE / 'contrast_v6_train.csv')
df = df.merge(df_contrast[['object_id','contrast']], on='object_id', how='left')
df['contrast_bin'] = pd.cut(df['contrast'], bins=[0,20,40,80,300],
    labels=['저대비(0~20)','기준(20~40)','중대비(40~80)','고대비(80+)'])
print(df['contrast_bin'].value_counts().sort_index())

fig, ax = plt.subplots(figsize=(9,3))
df['contrast'].hist(bins=60, ax=ax, color='steelblue', alpha=0.7)
ax.axvspan(20, 40, alpha=0.15, color='green', label='기준 품질(20~40)')
ax.axvspan(0,  20, alpha=0.15, color='red',   label='저대비(0~20) x3')
ax.set_xlabel('contrast (std)')
ax.set_title('학습 데이터 contrast 분포')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
df_low   = df[df['contrast'] <= 20]
df_train = pd.concat([df] + [df_low] * 2, ignore_index=True)
print(f'원본: {len(df)}건  확장후: {len(df_train)}건  (저대비 {len(df_low)}건 x3)')

## 3-1. 회전각 Pseudo Label 로드

high/medium 모두 사용(Fix 18: REC_HIGH_ONLY=False). `prepare_img()`에서 회전 보정에 사용.
이 라벨은 옛 브루트포스 각도 탐색(여러 각도로 돌려보고 OCR이 가장 잘 읽는 각도를 고름)으로 만든
추정값이라 완벽하지 않을 수 있음 — 남은 잔차 오차는 이후 `crop_text_region`(Fix 12: poly 각도로
재측정)과 `resolve_180_flip`(Fix 11: 0/180 sense 재확인)이 보완한다.

In [ ]:
# crop_v7_manifest.csv에서 object_id별 회전각/bbox 조회용 dict 생성
rot_map = df.set_index('object_id')['rotation_label_deg'].to_dict()
bbox_lookup_train = df.set_index('object_id')[['bbox_w', 'bbox_h']].to_dict('index')
print(f'rotation label 사용 가능: {len(rot_map)}건')
print('품질 분포(REC_HIGH_ONLY=False라 high+medium 둘 다 있어야 정상):')
print(df['rotation_label_quality'].value_counts())

## 4. 전처리 (고정 config)

In [ ]:
@dataclass
class PrepConfig:
    denoise_method: str = 'none'
    bilateral_d: int = 3
    bilateral_sigma_color: int = 20
    bilateral_sigma_space: int = 20
    nlm_h: int = 3
    clahe_clip: float = 2.5
    clahe_tile: int = 8
    unsharp_strength: float = 1.0
    unsharp_sigma: float = 1.5

DEFAULT_PREP = PrepConfig()

def preprocess_crop(image_bgr, cfg=DEFAULT_PREP):
    out = image_bgr
    if cfg.denoise_method == 'bilateral':
        out = cv2.bilateralFilter(out, cfg.bilateral_d,
                                  cfg.bilateral_sigma_color, cfg.bilateral_sigma_space)
    elif cfg.denoise_method == 'nlm':
        out = cv2.fastNlMeansDenoisingColored(out, None, cfg.nlm_h, cfg.nlm_h, 7, 21)
    gray     = cv2.cvtColor(out, cv2.COLOR_BGR2GRAY)
    clahe    = cv2.createCLAHE(clipLimit=cfg.clahe_clip,
                                tileGridSize=(cfg.clahe_tile, cfg.clahe_tile))
    enhanced = clahe.apply(gray)
    if cfg.unsharp_strength > 0:
        blurred  = cv2.GaussianBlur(enhanced, (0, 0), cfg.unsharp_sigma)
        enhanced = cv2.addWeighted(enhanced, 1+cfg.unsharp_strength, blurred, -cfg.unsharp_strength, 0)
    return cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)

def load_and_preprocess(crop_path, cfg=DEFAULT_PREP):
    img = cv2.imread(crop_path, cv2.IMREAD_UNCHANGED)
    if img is None: raise FileNotFoundError(crop_path)
    if img.ndim == 2: img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    elif img.shape[2] == 4: img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
    return preprocess_crop(img, cfg)

## 5. Augmentation 파이프라인

| 케이스 | 생성 이미지 수 |
|--------|---------------|
| 일반 | 1장 (detection 텍스트 크롭, Fix 5) |
| 혼동 문자(1/I/0/O/Q) | +1장 (랜덤 회전 jitter) |

RecAug(PaddleOCR 내장)가 추가 color/noise augmentation 담당. Fix 18: crop_v7 + `rotation_label_deg`로 원복 — `prepare_img`가 회전 보정(-angle) → 장축 정렬(90도 단위 마무리) → bbox 기준 업스케일까지 다시 수행한다. 이 라벨 기반 보정의 잔차 오차는 이후 Section 6-1의 `crop_text_region`(Fix 12)/`resolve_180_flip`(Fix 11)이 보완한다.

In [ ]:
def rotate_image(img, angle):
    if angle == 0: return img
    h, w = img.shape[:2]
    cx, cy = w/2, h/2
    M = cv2.getRotationMatrix2D((cx, cy), -angle, 1.0)
    cos, sin = abs(M[0,0]), abs(M[0,1])
    new_w = int(h*sin + w*cos)
    new_h = int(h*cos + w*sin)
    M[0,2] += (new_w/2) - cx
    M[1,2] += (new_h/2) - cy
    return cv2.warpAffine(img, M, (new_w, new_h),
                          flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_REPLICATE)  # Fix 3: 검정 패딩→엣지 오인식 방지, 경계 픽셀 복제

def align_to_long_axis(img):
    """세로로 긴 이미지 → 90° 회전해 가로로 맞춤 (이미지 실제 크기 기준).
    Fix 18: crop_v7 원복으로 재도입 — rotation_label_deg가 대략 눕혀줘도 90도 단위로
    남는 경우가 있어 이걸로 마무리 정렬."""
    h, w = img.shape[:2]
    if h > w * 1.2:
        return rotate_image(img, 90)
    return img

def upscale_if_small(img, bbox_w, bbox_h, area_thresh=71818, scale=2.0):
    """작은 crop → 2배 확대 (Fix 18: crop_v7 매니페스트의 bbox_w/h 기준으로 복원)."""
    if bbox_w * bbox_h <= area_thresh:
        h, w = img.shape[:2]
        img = cv2.resize(img, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_CUBIC)
    return img

def prepare_img(crop_path, bbox_w, bbox_h, rot_angle=None):
    """공통 전처리 파이프라인: preprocess → rotate(−angle) → align → upscale
    train/val 양쪽 동일 순서 보장 (Fix 3). Fix 18: crop_v7 + rotation_label_deg로 원복.
    이 회전 보정은 추정값이라 잔차가 남을 수 있음 — 이후 crop_text_region(Fix 12)의
    poly 각도 재측정과 resolve_180_flip(Fix 11)이 보완한다."""
    img = load_and_preprocess(crop_path)
    if rot_angle is not None:
        # rotation_label_deg = 알약의 현재 기울기 → 보정은 반대 방향(-angle)
        img = rotate_image(img, (-int(rot_angle)) % 360)
    img = align_to_long_axis(img)
    img = upscale_if_small(img, bbox_w, bbox_h)
    return img

CONFUSION_CHARS = set('01IOQ')

def augment_image(base_img, rng, target_text=''):
    """혼동 문자(0/1/I/O/Q) 포함 시 소폭 jitter 회전 이미지 추가 (Fix 6).
    회전 보정은 prepare_img/crop_text_region에서 이미 완료된 상태로 호출됨."""
    results = [base_img]
    if CONFUSION_CHARS & set(target_text):
        jitter = float(rng.uniform(-8, 8))
        results.append(rotate_image(base_img, jitter))
    return results

## 6. Paddle 환경 확인

In [ ]:
import paddle
print(f'PaddlePaddle: {paddle.__version__}')
print(f'GPU: {paddle.is_compiled_with_cuda()}  count={paddle.device.cuda.device_count()}')
import paddleocr
print(f'PaddleOCR: {paddleocr.__version__}')  # 학습·평가 세션 버전 기록 (재현성)

## 6-1. Detection 기반 텍스트 영역 크롭 (Fix 5) + 임의 회전 보정 (Fix 12) + 180도 보정 (Fix 11) + eval 전용 multi-angle (Fix 23)

recognition 학습/내부 eval용 이미지를 만들 때, 알약 crop 전체가 아니라 **detection이 찾은 실제 각인 텍스트 영역만** 잘라서 쓴다. eval(Section 12)이 원래 det+rec 풀 파이프라인이므로, train도 같은 입력 분포(텍스트 영역만)로 맞추는 것.

**Fix 18 (crop_v7 원복)**: v8/v9 초반에 시도했던 YOLO OBB straighten crop을 그만두고 crop_v7 + `rotation_label_deg`(옛 브루트포스 각도 탐색 라벨)로 되돌아감(pretrained 모델까지 같이 무너진 걸로 봐서 OBB crop 자체의 문제로 판단). `prepare_img`가 1차로 이 라벨 기반 회전 보정을 수행한다.

**Fix 12는 이제 '보완' 역할**: `rotation_label_deg`는 추정값이라 잔차 오차가 남을 수 있음(탐색 각도 간격, 흐린 각인으로 인한 오판 등). 여기서는 알약이 아니라 **감지된 텍스트 poly 자체의 장축각**으로 다시 한번 회전 보정을 한다 — 라벨이 대략 맞춰놓은 상태에서 detection이 여전히 텍스트를 찾을 수 있으면, 실측 각도로 미세 조정해서 라벨의 정밀도 한계를 보완한다. 멀티라인(박스 2개 이상)이면 poly별 각도를 면적 가중 원형평균으로 묶어 대표각을 쓴다. 모양만으로는 180도 sense까지는 못 구분하므로, 그 마지막 단계는 아래 Fix 11이 처리한다.

**Fix 17**: `rotation_label_deg` 보정 후에도 detection이 아예 박스를 못 찾는 경우(라벨 오차가 detection 관용범위를 벗어날 만큼 큰 경우)에는 90/180/270도로 재시도한다. 최초 detection 실패 케이스에만 적용되고 모든 crop에 일괄 적용하지 않는다.

**Fix 11**: `rotation_label_deg` + `align_to_long_axis` + Fix 12로 정렬을 다 마쳐도, 180° 뒤집힘 자체는 라벨링 과정에서 방향 판단이 틀렸을 가능성이 남는다(추정 각도가 180도 밀려도 poly 모양은 똑같아 보임). 그래서 별도 textline orientation 분류기로 0/180 sense를 마지막에 한 번 더 확인해 바로잡는다.

**남은 한계**: Fix 17까지도 detection이 텍스트를 찾아낸 경우에만 도움이 된다. detection 자체가 실패하는 케이스는 Fix 14(전체 crop fallback)로 학습/eval에는 포함되지만 품질은 낮은 채로 남는다.

**Fix 22→23 (Section 12 전용 전처리 재검증)**: 위 `crop_text_region`은 Section 9(학습데이터 생성)처럼 recognition 전용 모델(내장 detection 없음)에 넣을 이미지를 만들 때는 그대로 필요하다. Section 12 eval은 detection까지 내장된 풀 `PaddleOCR()` 파이프라인이라 미리 타이트하게 잘라서 넘기면 오히려 손해라는 게 별도 ablation(`ys_pretrained_ocr_score_v7_preprocessing_ablation*.ipynb`)으로 확인됨(Fix 22) — 크롭 없이 회전만 보정하는 `rotate_only()`로 처음 교체했었다. 그런데 `ablation_v2`에서 크롭을 완전히 뺀 채로 `rotate_only`(poly 각도 미세보정) 단독만 적용해도 baseline(아무 후처리 없음)보다 더 나쁘게 나옴(EM -0.056) — 즉 문제는 크롭뿐 아니라 poly 기반 미세 회전보정 자체에도 있었다(해당 arm은 `resolve_180_flip`을 호출하지 않았고, 180도 처리는 baseline을 포함한 모든 arm이 `use_textline_orientation=True` 내장 처리에만 의존했음 — `resolve_180_flip` 자체가 나쁘다고 확인된 적은 없고, 3-way/4-way ablation에서 크롭 위에 얹었을 때는 오히려 근소하게 긍정적이었음). 반면 30도 단위 굵은 각도 후보로 통째로 돌려보고 confidence로 고르는 multi-angle 탐색 자체는 적용 여부 비교에서 유의미하게 도움이 되는 것으로 확인됨. 최종적으로 `rotate_only()` 함수 자체를 제거하고, `resolve_rotation_by_confidence`는 30도 단위 `rotate_image` 결과를 추가 보정 없이 그대로 OCR에 넣어 confidence로 최적 각도만 고르는 순수 multi-angle 탐색만 남겼다(Fix 23; `resolve_180_flip`도 같이 뺐지만 이는 '나쁘다고 확인돼서'가 아니라 근거 없는 추가 호출 비용을 줄이기 위한 판단). Section 9/`setup_local_data`는 `crop_text_region`을 그대로 유지한다.


In [ ]:
from paddleocr import TextDetection

det_model = TextDetection(model_name='PP-OCRv5_server_det')


def _get_det_polys(det_result_item):
    for key in ('dt_polys', 'det_polys', 'boxes'):
        if key in det_result_item:
            return det_result_item[key]
    return []


def _long_axis_angle_deg(pts):
    """poly 4점(픽셀) → 장축각(도). 팀원의 OBB 장축각 계산과 동일한 방식
    (인접 두 변 중 더 긴 쪽을 장축으로 보고 그 방향의 각도를 구함)."""
    e12, e23 = pts[1] - pts[0], pts[2] - pts[1]
    l12, l23 = float(np.hypot(*e12)), float(np.hypot(*e23))
    long_e, L, S = (e12, l12, l23) if l12 >= l23 else (e23, l23, l12)
    theta = float(np.degrees(np.arctan2(long_e[1], long_e[0])))
    return theta, L, S


def _run_detection(img):
    """det_model로 poly 리스트만 뽑아주는 헬퍼 (Fix 17 재시도 로직에서 재사용)."""
    result = list(det_model.predict(img))
    if not result:
        return []
    return _get_det_polys(result[0])


def crop_text_region(img, padding_ratio=0.12, min_area=64, single_only=False):
    """detection으로 텍스트 영역을 찾아 그 부분만 크롭.
    Fix 9: v6에서 single_only=True(멀티라인 완전 제외)를 기본값으로 했더니, val에는
    여전히 존재하는 멀티라인 각인에 대해 모델이 아예 학습 노출이 없어져 EM이 오히려
    하락했음(0.683→0.665). '노이즈를 감수하고 커버리지를 유지'하기로 결정 —
    박스 2개 이상이면 전체를 감싸는 bbox로 병합해서 학습에 포함(v4와 동일 방식).
    single_only=True로 주면 v5/v6처럼 멀티라인을 완전히 제외(비교/실험용).

    Fix 12 (Fix 18: crop_v7 원복 후 '보완' 역할로 재해석): `prepare_img`의 `rotation_label_deg`
    기반 회전 보정은 추정값이라 잔차 오차가 남을 수 있다. 여기서는 알약 모양이 아니라 감지된
    텍스트 poly 자체의 장축각으로 다시 한번 회전 보정을 해서, 라벨이 대략 맞춰놓은 상태를
    실측 각도로 미세 조정한다. 다만 모양만으로는 180도 sense(뒤집힘)까지는 못 구분하므로 그건
    그대로 resolve_180_flip이 처리한다. 박스가 여러 개(멀티라인)면 poly별 각도를 면적 가중
    원형평균(circular mean)으로 묶어 대표 회전각을 쓴다.

    Fix 17 (9-2 진단 결과 반영): 최초 detection이 박스를 하나도 못 찾은 경우에만
    90/180/270도로 돌려가며 재시도한다. 9-2 진단에서 detection 실패 샘플의 과반(56%)이
    회전만 시키면 성공하는 것으로 확인되어 추가함. 정상적으로 박스가 잡힌 케이스(단순히
    single_only 조건 때문에 제외되는 경우 포함)는 재시도 비용을 들이지 않는다 — 모든 crop에
    일괄 적용하지 않고 실패 케이스에만 적용.
    반환: (크롭 이미지 또는 None, 검출된 박스 수)"""
    polys = _run_detection(img)
    n = len(polys)

    if n == 0:
        for _retry_angle in (90, 180, 270):
            _rimg = rotate_image(img, _retry_angle)
            _rpolys = _run_detection(_rimg)
            if len(_rpolys) > 0:
                img, polys, n = _rimg, _rpolys, len(_rpolys)
                break

    if n == 0:
        return None, n
    if single_only and n != 1:
        return None, n

    polys = [np.asarray(p, dtype=np.float64) for p in polys]

    # Fix 12: poly 장축각 → 면적 가중 원형평균 → ±90도 접기
    angles_rad, areas = [], []
    for p in polys:
        theta, L, S = _long_axis_angle_deg(p)
        angles_rad.append(np.radians(theta))
        areas.append(max(L * S, 1e-6))
    areas = np.asarray(areas)
    mean_sin = float(np.sum(np.sin(angles_rad) * areas) / areas.sum())
    mean_cos = float(np.sum(np.cos(angles_rad) * areas) / areas.sum())
    theta = float(np.degrees(np.arctan2(mean_sin, mean_cos)))
    rot = (theta + 90.0) % 180.0 - 90.0

    pts_all = np.concatenate(polys, axis=0)  # (N, 2), 회전 전 좌표
    H, W = img.shape[:2]
    xs0, ys0 = pts_all[:, 0], pts_all[:, 1]
    cx, cy = float((xs0.min() + xs0.max()) / 2), float((ys0.min() + ys0.max()) / 2)

    if abs(rot) > 1.0:  # 1도 이하 미세각은 보간 손실 방지 위해 생략
        Mrot = cv2.getRotationMatrix2D((cx, cy), rot, 1.0)
        img = cv2.warpAffine(img, Mrot, (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
        ones = np.ones((pts_all.shape[0], 1))
        pts_all = (Mrot @ np.concatenate([pts_all, ones], axis=1).T).T  # poly 좌표도 같이 회전

    xs, ys = pts_all[:, 0], pts_all[:, 1]
    x0, x1 = float(xs.min()), float(xs.max())
    y0, y1 = float(ys.min()), float(ys.max())
    if (x1 - x0) * (y1 - y0) < min_area:
        return None, n

    pad_x = (x1 - x0) * padding_ratio
    pad_y = (y1 - y0) * padding_ratio
    x0 = max(0, int(x0 - pad_x)); x1 = min(W, int(x1 + pad_x))
    y0 = max(0, int(y0 - pad_y)); y1 = min(H, int(y1 + pad_y))
    if x1 <= x0 or y1 <= y0:
        return None, n
    return img[y0:y1, x0:x1], n


# API 응답 구조 확인 (샘플 1장)
# dt_polys가 실제 키 이름이 아니면 위 _get_det_polys의 후보 목록에 실제 키를 추가할 것
_probe_path = next(CROP_DIR.glob('*.png'))
_probe_img = load_and_preprocess(str(_probe_path))
_probe_result = list(det_model.predict(_probe_img))
if _probe_result:
    print('detection 응답 키:', list(_probe_result[0].keys()))
    print('감지된 박스 수:', len(_get_det_polys(_probe_result[0])))
else:
    print('[WARN] 샘플 이미지에서 detection 결과가 비어있음')

_probe_crop, _probe_n = crop_text_region(_probe_img)
print('크롭 성공 여부:', _probe_crop is not None,
      f'| 박스 {_probe_n}개 | shape={_probe_crop.shape}' if _probe_crop is not None else f'| 박스 {_probe_n}개')


# ── Fix 11: 180도 보정용 textline orientation 분류기 ──────────────────────────
# 클래스명이 이 PaddleOCR 버전에서 다를 수 있음 — 아래 프린트로 실제 키/값 확인해서 필요하면 조정할 것.
_textline_ori_available = False
try:
    from paddleocr import TextLineOrientationClassification
    textline_ori_model = TextLineOrientationClassification(model_name='PP-LCNet_x1_0_textline_ori')
    _textline_ori_available = True
except Exception as _e:
    print(f'[WARN] TextLineOrientationClassification 로드 실패({_e}) — 180도 보정 없이 진행합니다. '
          f'클래스명이 다르면 이 셀만 고쳐서 다시 실행할 것.')


def _get_ori_label(item):
    for key in ('label_names', 'class_ids', 'scores'):
        if key in item:
            return item[key]
    return None


def resolve_180_flip(crop):
    """텍스트 영역 crop이 180도 뒤집혔으면 바로잡는다. 판별 실패/모델 없음이면 원본 그대로 반환.
    prepare_img(rotation_label_deg) + crop_text_region(Fix 12)이 수평 정렬까지 끝내고 넘겨주므로,
    남은 건 0/180 sense뿐 — 라벨링 과정에서 방향 판단이 틀렸을 가능성을 여기서 마지막으로 잡는다."""
    if not _textline_ori_available or crop is None:
        return crop
    try:
        result = list(textline_ori_model.predict(crop))
        if not result:
            return crop
        labels = _get_ori_label(result[0])
        label_str = str(labels[0]) if isinstance(labels, (list, tuple)) and labels else str(labels)
        if '180' in label_str:
            return rotate_image(crop, 180)
        return crop
    except Exception:
        return crop


# API 응답 구조 확인 (샘플 1장) — label_names 실제 키/값 형식이 다르면 _get_ori_label 수정 필요
if _textline_ori_available and _probe_crop is not None:
    _probe_ori_result = list(textline_ori_model.predict(_probe_crop))
    if _probe_ori_result:
        print('textline orientation 응답 키:', list(_probe_ori_result[0].keys()))
        print('textline orientation 라벨:', _get_ori_label(_probe_ori_result[0]))

## 7. 평가용 매칭 함수 정의 (`score_one` / `exact_match`)

여기 정의된 관대한(substring) 매칭은 **Section 12 평가 전용**. 학습 데이터 채택에는 더 이상 관여하지 않음 (Section 2-1의 `train_rec_label_v3.txt` 직접 로드로 대체, Fix 2).

In [ ]:
def levenshtein(pred, target):
    p, t = list(pred), list(target)
    dp = list(range(len(t)+1))
    for pc in p:
        ndp = [dp[0]+1]
        for j, tc in enumerate(t):
            ndp.append(min(dp[j]+(pc!=tc), dp[j+1]+1, ndp[-1]+1))
        dp = ndp
    return dp[len(t)]

def score_one(pred, candidates):
    """Fix 10: substring 관대 매칭(짧은 candidate가 pred에 우연히 포함되면 만점 처리) 제거.
    crop_v7은 이미 '마크 포함 행'을 전부 걸러낸 데이터라, 그 escape가 정당화되는 케이스 자체가
    없음 — 있으나 마나 한 조건을 남겨두는 대신 아예 삭제하고 순수 CER 기반 점수만 반환."""
    pred  = normalize_imprint(pred)
    valid = [c for c in candidates if c]
    if not valid: return None
    best_cer = min(levenshtein(pred, c)/max(len(c),1) for c in valid)
    return max(0.0, 1.0 - best_cer)

def exact_match(pred, candidates):
    """Fix 10: substring 관대 매칭 제거, 정확히 일치하는 경우만 True."""
    pred = normalize_imprint(pred)
    return any(pred == c for c in candidates if c)

def _extract_ocr(page):
    texts = [str(t) for t in page.get('rec_texts', [])]
    confs = [float(c) for c in page.get('rec_scores', [])]
    polys = page.get('rec_polys', [])
    if not texts: return '', '', float('nan'), [], []
    if polys and len(polys) == len(texts):
        def _cy(poly): return sum(p[1] for p in poly)/len(poly)
        def _cx(poly): return sum(p[0] for p in poly)/len(poly)
        heights = [max(p[1] for p in poly)-min(p[1] for p in poly) for poly in polys]
        row_thresh = (sum(heights)/len(heights))/2
        items = sorted(zip(texts, polys, confs),
                       key=lambda x: (round(_cy(x[1])/row_thresh), _cx(x[1])))
        texts = [t for t,_,_ in items]
        polys = [p for _,p,_ in items]
        confs = [c for _,_,c in items]
    return (' | '.join(texts), normalize_imprint(''.join(texts).strip()),
            float(np.mean(confs)), polys, confs)

## 9. Recognition Fine-tuning 데이터 생성

**scratch 디스크(/mnt/local-scratch, ~350GB)에 직접 쓰기** → /content 디스크 용량 초과 방지.  
포맷: `crops/xxx.png<TAB>각인텍스트`

정답은 Section 7의 `answer_map`(object_id → 단일 정답, 이미 눈으로 검증됨)을 그대로 사용. **Fix 5**: 알약 crop 전체가 아니라 Section 6-1의 `crop_text_region()`으로 실제 각인 텍스트 영역만 잘라서 저장 — eval(Section 12)이 항상 det+rec 풀 파이프라인이었던 것과 입력 분포를 맞추기 위함. 증강본 파일명(`{object_id}_{idx}_{face}.png`)은 학습용 `train_rec_label.txt`에만 쓰고, 검증 완료된 원본 매핑 `train_rec_label_v3.txt`는 Section 11에서도 절대 덮어쓰지 않음(Fix 1).

In [ ]:
# Scratch 디스크 설정 (없으면 /content 사용)
scratch_base = Path('/mnt/local-scratch')
if scratch_base.exists():
    SCRATCH_CROPS = scratch_base / 'crops'
    print(f'Scratch 디스크 사용: {scratch_base}')
    !df -h /mnt/local-scratch
else:
    SCRATCH_CROPS = REC_TRAIN_DIR / 'crops'
    print('[WARN] /mnt/local-scratch 없음 → /content 사용 (236GB, 용량 주의)')
    !df -h /content

SCRATCH_CROPS.mkdir(parents=True, exist_ok=True)
rec_crops_link = REC_TRAIN_DIR / 'crops'
if scratch_base.exists():  # scratch 쓸 때만 symlink
    if rec_crops_link.is_symlink(): rec_crops_link.unlink()
    elif rec_crops_link.exists(): shutil.rmtree(str(rec_crops_link))
    rec_crops_link.symlink_to(SCRATCH_CROPS)
    print(f'심링크: {rec_crops_link} -> {SCRATCH_CROPS}')
print(f'기존 이미지: {len(list(SCRATCH_CROPS.iterdir()))}개')

### 9-1. 최종 학습 입력 눈검증

2-1-1의 눈검증은 원본 crop + 정답 텍스트만 확인했음. 실제로 recognition 모델에 들어가는 건 `prepare_img(rotation_label_deg 보정) → crop_text_region(detection 크롭 + Fix 12 잔차 보정) → resolve_180_flip(Fix 11)`을 거친 결과라, 그 최종형을 따로 확인한다. 텍스트가 거꾸로 보이면 180도 보정이 실패한 것이고, 기울어져 있으면 `rotation_label_deg` 자체가 크게 틀렸을 가능성이 크다. **여기서 이상하면 Section 9로 넘어가지 말 것.**

In [ ]:
sample = df_train.sample(min(16, len(df_train)), random_state=0)
fig, axes = plt.subplots(4, 4, figsize=(14, 8))
for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    obj_id = str(row['object_id'])
    bbox_w = float(row.get('bbox_w', 0) or 0)
    bbox_h = float(row.get('bbox_h', 0) or 0)
    img = prepare_img(row['crop_path'], bbox_w, bbox_h, rot_angle=rot_map.get(obj_id))
    text_crop, n_boxes = crop_text_region(img)
    ttext = answer_map.get(obj_id, '?')
    if text_crop is not None:
        text_crop = resolve_180_flip(text_crop)
        ax.imshow(cv2.cvtColor(text_crop, cv2.COLOR_BGR2RGB))
        ax.set_title(ttext, fontsize=9)
    else:
        # 실패 원인을 눈으로 바로 보게 원본(straighten crop) 이미지를 그대로 보여줌
        reason = '박스 0개' if n_boxes == 0 else f'박스 {n_boxes}개(제외)'
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f'{reason} | GT: {ttext}', fontsize=8, color='red')
    ax.axis('off')
plt.suptitle('학습 입력 최종형 — 텍스트가 수평 한 줄, 뒤집히지 않았고 정답과 일치하는가')
plt.tight_layout()
plt.show()
print('▲ 텍스트가 기울어져 있거나(원형 알약이면 알려진 한계), 거꾸로거나, 딴 면 정답이 붙어 있으면 원인부터 잡을 것.')

In [ ]:
rng = np.random.default_rng(seed=42)
rec_label_lines = []
write_errors = 0
skip_no_answer = 0
skip_no_det = 0
skip_multi_box = 0
fallback_whole_crop = 0

for idx, row in tqdm(df_train.iterrows(), total=len(df_train), desc='Rec 학습데이터'):
    obj_id = str(row['object_id'])
    ttext  = answer_map.get(obj_id)
    if not ttext:
        skip_no_answer += 1
        continue

    bbox_w = float(row.get('bbox_w', 0) or 0)
    bbox_h = float(row.get('bbox_h', 0) or 0)
    # Fix 18: crop_v7 + rotation_label_deg로 원복. prepare_img가 1차 회전 보정 수행
    img_proc = prepare_img(row['crop_path'], bbox_w, bbox_h, rot_angle=rot_map.get(obj_id))
    # Fix 5: 알약 전체가 아니라 detection이 찾은 텍스트 영역만 잘라서 recognition 학습에 사용
    # Fix 9: 박스가 2개 이상(여러 줄)이어도 병합해서 학습에 포함 (single_only=False 기본값)
    # Fix 12/17: poly 각도로 잔차 보정 + 최초 실패 시 90/180/270도 재시도
    text_crop, n_boxes = crop_text_region(img_proc)
    if text_crop is None:
        # Fix 14: detection 실패라고 통째로 제외하지 않고 전체 crop을 그대로 fallback으로 사용.
        # 최종 eval(Section 12)은 detection 실패 케이스도 채점 대상에 포함하는 전체 val 집합
        # 기준인데, 여기서 실패 샘플을 계속 빼면 학습/내부 eval은 '쉬운 부분집합' 기준이 되어
        # best_accuracy 체크포인트 선택 자체가 왜곡됨(코드 리뷰 지적). 완벽히 정렬 안 된 채로라도
        # 학습에 포함시켜 최종 평가 모집단과 맞추는 쪽을 택함.
        if n_boxes == 0:
            skip_no_det += 1
        else:
            skip_multi_box += 1
        text_crop = img_proc
        fallback_whole_crop += 1
    # Fix 11: 텍스트 영역이 180도 뒤집혔으면 학습 라벨과 짝지어지기 전에 바로잡음
    # (fallback인 경우 poly 정보가 없어 완전한 정렬은 보장 못 하지만 best-effort로 시도)
    text_crop = resolve_180_flip(text_crop)

    aug_imgs = augment_image(text_crop, rng, target_text=ttext)
    for ai, aug_img in enumerate(aug_imgs):
        fname = f'{obj_id}_{idx}_{ai}.png'
        ok = cv2.imwrite(str(SCRATCH_CROPS / fname), aug_img)
        if ok:
            rec_label_lines.append(f'crops/{fname}	{ttext}')
        else:
            write_errors += 1
            if write_errors <= 5:
                print(f'[WARN] imwrite 실패: {fname} (디스크 용량 확인 필요)')

(REC_TRAIN_DIR/'train_rec_label.txt').write_text('\n'.join(rec_label_lines), encoding='utf-8')
print(f'Recognition label: {len(rec_label_lines)}건  쓰기 오류: {write_errors}건')
print(f'정답 매핑 없어서 제외: {skip_no_answer}건')
print(f'detection 실패 → 전체 crop fallback 사용: {fallback_whole_crop}건 '
      f'(no_det={skip_no_det}, multi_box={skip_multi_box}) — Fix 14: 더 이상 학습에서 제외하지 않음')
# Fix 14(P2-2): batch_size/drop_last로 매 epoch 버려지는 잔여 샘플 수 확인
_BATCH_SIZE = 256
_n_train = len(rec_label_lines)
print(f'train 샘플 {_n_train}개, batch_size={_BATCH_SIZE} 기준 매 epoch drop_last로 버려지는 잔여: '
      f'{_n_train % _BATCH_SIZE}개 ({(_n_train % _BATCH_SIZE) / max(_n_train,1) * 100:.2f}%)')
!df -h /mnt/local-scratch

### 9-2. detection 실패 원인 진단 (회전 문제 vs 다른 이유)

`detection 실패로 제외` 건수가 왜 생기는지 원인을 나누어 본다. 실패한 샘플을 90/180/270도로 돌려서 detection을 재시도했을 때 성공하면 **회전 문제**로, 그래도 실패하면 **다른 이유(흐림, crop이 너무 작음, 각인 자체가 옅음 등)** 로 분류한다. 여기서 회전 문제 비중이 크게 나오면, `crop_text_region`에 다각도 재시도 fallback을 추가하는 게 근본적으로 도움이 되고, 비중이 작으면 다른 원인(흐림/크기)부터 봐야 한다. **학습 파이프라인에는 영향 없는 1회성 진단 셀** — 결과만 확인하고 넘어갈 것.

In [ ]:
import pandas as pd

def _blur_score(img_bgr):
    """라플라시안 분산 — 값이 낮을수록 흐릿함."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())

N_DIAG = min(300, len(df_train))  # 전체 다 돌면 오래 걸리니 샘플만 진단
diag_rows = []
for idx, row in tqdm(df_train.sample(N_DIAG, random_state=0).iterrows(), total=N_DIAG, desc='detection 실패 진단'):
    obj_id = str(row['object_id'])
    if not answer_map.get(obj_id):
        continue
    bbox_w = float(row.get('bbox_w', 0) or 0)
    bbox_h = float(row.get('bbox_h', 0) or 0)
    img = prepare_img(row['crop_path'], bbox_w, bbox_h, rot_angle=rot_map.get(obj_id))
    text_crop, n_boxes = crop_text_region(img)
    if text_crop is not None:
        continue  # 성공 케이스는 진단 대상 아님

    h, w = img.shape[:2]
    rotation_fixed = False
    for test_angle in (90, 180, 270):
        rimg = rotate_image(img, test_angle)
        rresult = list(det_model.predict(rimg))
        if rresult and len(_get_det_polys(rresult[0])) > 0:
            rotation_fixed = True
            break

    diag_rows.append({
        'object_id': obj_id, 'w': w, 'h': h, 'area': w * h,
        'blur_score': _blur_score(img), 'rotation_fixed': rotation_fixed,
        'n_boxes_original': n_boxes,
    })

df_diag = pd.DataFrame(diag_rows)
print(f'진단 대상(detection 실패) 샘플: {len(df_diag)}건 / 전체 샘플링 {N_DIAG}건')
if len(df_diag) > 0:
    print(f'회전하면 detection 성공(=회전 문제로 추정): {df_diag["rotation_fixed"].sum()}건 '
          f'({df_diag["rotation_fixed"].mean()*100:.1f}%)')
    print(f'회전해도 실패 유지(=다른 이유로 추정): {(~df_diag["rotation_fixed"]).sum()}건')
    print()
    print('[회전 여부별 crop 면적/흐림도 비교 — 계속 실패하는 쪽이 유독 작거나 흐리면 그게 진짜 원인]')
    print(df_diag.groupby('rotation_fixed')[['area', 'blur_score']].describe())
else:
    print('샘플 안에 detection 실패 케이스가 없음 — N_DIAG를 늘려서 다시 시도할 것')

# 눈검증: 카테고리별 샘플
path_map = dict(zip(df_train['object_id'].astype(str), df_train['crop_path']))
for label, sub_df in [('회전하면 성공', df_diag[df_diag['rotation_fixed']] if len(df_diag) else df_diag),
                       ('회전해도 실패', df_diag[~df_diag['rotation_fixed']] if len(df_diag) else df_diag)]:
    if len(sub_df) == 0:
        print(f'[{label}] 해당 샘플 없음')
        continue
    sample = sub_df.sample(min(6, len(sub_df)), random_state=0)
    fig, axes = plt.subplots(1, len(sample), figsize=(3 * len(sample), 3))
    if len(sample) == 1:
        axes = [axes]
    for ax, (_, r) in zip(axes, sample.iterrows()):
        _oid = r['object_id']
        _bb = bbox_lookup_train.get(_oid, {})
        img = prepare_img(path_map[_oid], float(_bb.get('bbox_w', 0) or 0),
                          float(_bb.get('bbox_h', 0) or 0), rot_angle=rot_map.get(_oid))
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        title = f"blur={r['blur_score']:.0f} / area={r['area']:.0f}"
        ax.set_title(title, fontsize=8)
        ax.axis('off')
    plt.suptitle(label)
    plt.tight_layout()
    plt.show()

### 9-3. val 데이터도 학습에 편입 (v10, Fix 24)


In [ ]:
# Fix 24: 팀 결정으로 val split도 학습 데이터에 포함시킨다. train과 동일한 파이프라인
# (prepare_img -> crop_text_region -> resolve_180_flip -> augment_image)으로 crop_v7_val.zip +
# val_rec_label_v3.txt를 처리해서 위에서 만든 train_rec_label.txt에 이어붙인다.
# 주의(팀 합의로 감수하기로 한 leakage): Section 11(setup_local_data)이 만드는 rec_val/Eval과
# Section 12(최종 eval)는 이 셀 이후에도 코드 변경 없이 그대로 val_rec_label_v3.txt 기준으로
# 채점하는데, 그 이미지들이 이제 Train에도 들어있으므로 두 지표 모두 순수한 held-out 성능이
# 아니라 부분적으로 leakage된 값이다. 신규 holdout을 따로 떼지 않고 이 leakage를 그대로
# 감수하기로 함(팀 결정) -- v10의 Section 12 숫자는 v9까지의 honest 숫자와 비교 불가.
import shutil as _shutil9, zipfile as _zf9

VAL_CROP_ZIP_9 = DRIVE_BASE / 'crop_v7_val.zip'
VAL_CROP_DIR_9 = Path('/content/crops_v7_val')
if VAL_CROP_ZIP_9.exists():
    if not VAL_CROP_DIR_9.exists() or not any(VAL_CROP_DIR_9.iterdir()):
        print('val crop 압축 해제 중...')
        with _zf9.ZipFile(VAL_CROP_ZIP_9, 'r') as zf:
            zf.extractall(VAL_CROP_DIR_9)
    print(f'val raw crop: {sum(1 for _ in VAL_CROP_DIR_9.iterdir())}개')
else:
    VAL_CROP_DIR_9 = CROP_DIR
    print('[WARN] crop_v7_val.zip 없음 -> train crop 폴더 사용')

df_val_for_train9 = pd.read_csv(MANIFEST_CSV, low_memory=False)
df_val_for_train9 = df_val_for_train9[df_val_for_train9['split'] == 'val']
rot_map_val9 = df_val_for_train9.set_index('object_id')['rotation_label_deg'].to_dict()
bbox_lookup_val9 = df_val_for_train9.set_index('object_id')[['bbox_w', 'bbox_h']].to_dict('index')

val_answer_lines9 = (DRIVE_BASE / 'val_rec_label_v3.txt').read_text(encoding='utf-8').strip().split('\n')
val_answer_map9 = {}
for line in val_answer_lines9:
    parts = line.split('\t')
    if len(parts) != 2:
        continue
    oid = Path(parts[0]).stem
    val_answer_map9[oid] = parts[1].strip()
print(f'val_rec_label_v3.txt 로드: {len(val_answer_map9)}개 object_id')

val_skip_no_img, val_skip_no_det, val_skip_multi_box, val_fallback_whole_crop, val_added = 0, 0, 0, 0, 0
for oid, ttext in tqdm(val_answer_map9.items(), desc='val -> train 편입'):
    src9 = VAL_CROP_DIR_9 / f'{oid}.png'
    if not src9.exists():
        val_skip_no_img += 1
        continue
    bb9 = bbox_lookup_val9.get(oid, {})
    bbox_w9 = float(bb9.get('bbox_w', 0) or 0)
    bbox_h9 = float(bb9.get('bbox_h', 0) or 0)
    img_proc9 = prepare_img(str(src9), bbox_w9, bbox_h9, rot_angle=rot_map_val9.get(oid))
    text_crop9, n_boxes9 = crop_text_region(img_proc9)
    if text_crop9 is None:
        if n_boxes9 == 0:
            val_skip_no_det += 1
        else:
            val_skip_multi_box += 1
        text_crop9 = img_proc9
        val_fallback_whole_crop += 1
    text_crop9 = resolve_180_flip(text_crop9)

    aug_imgs9 = augment_image(text_crop9, rng, target_text=ttext)
    for ai9, aug_img9 in enumerate(aug_imgs9):
        fname9 = f'{oid}_val_{ai9}.png'
        ok9 = cv2.imwrite(str(SCRATCH_CROPS / fname9), aug_img9)
        if ok9:
            rec_label_lines.append(f'crops/{fname9}\t{ttext}')
            val_added += 1
        else:
            print(f'[WARN] imwrite 실패: {fname9}')

(REC_TRAIN_DIR/'train_rec_label.txt').write_text('\n'.join(rec_label_lines), encoding='utf-8')
print(f'val -> train 편입 완료: {val_added}건 추가 (총 train 라인: {len(rec_label_lines)}건)')
print(f'val 이미지 없어서 스킵: {val_skip_no_img}건, detection 실패 fallback: {val_fallback_whole_crop}건 '
      f'(no_det={val_skip_no_det}, multi_box={val_skip_multi_box})')


## 10. Fine-tuning Config 생성

**변경사항 (v1 대비)**  
- Det: `EastRandomCropData` 포함, `eval_batch_step [0, 2000]`, `Eval:` 섹션 포함  
- Rec: `RecConAug` 제거 (ext_data 에러 원인), `eval_batch_step [0, 1500]`, `Eval:` 섹션 포함  
- 두 config 모두 pretrained_model 경로 `/content/pretrain_models/...` 직접 지정

In [ ]:
rec_yaml = """
Global:
  use_gpu: true
  epoch_num: 20
  log_smooth_window: 20
  print_batch_step: 10
  save_model_dir: ./output/rec_finetune/
  save_epoch_step: 1
  eval_batch_step: [0, 150]
  cal_metric_during_train: true
  pretrained_model: /content/pretrain_models/PP-OCRv5_server_rec_pretrained
  character_dict_path: /content/ppocrv5_dict.txt  # Fix 16: Section 7의 OOV 체크에 쓴 파일과 동일 (git clone 사본 아님)
  use_visualdl: false
  max_text_length: 25
  use_space_char: true
  use_amp: true
  scale_loss: 1024.0

Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: 0.0001
    warmup_epoch: 4
  regularizer:
    name: L2
    factor: 0.00003

Architecture:
  model_type: rec
  algorithm: SVTR_HGNet
  Backbone:
    name: PPHGNetV2_B4
    text_rec: true
    # backbone freeze는 이 PaddleOCR 버전/백본에서 실제 지원 여부가 확인 안 되어 제외.
    # 대신 낮은 lr(0.00005)·짧은 epoch(5)를 catastrophic forgetting 방지용 안전장치로 사용.
  Head:
    name: MultiHead
    head_list:
      - CTCHead:
          Neck:
            name: svtr
            dims: 120
            depth: 2
            hidden_dims: 120
            kernel_size: [1, 3]
            use_guide: true
          Head:
            fc_decay: 0.00001
      - NRTRHead:
          nrtr_dim: 384
          max_text_length: 25

Loss:
  name: MultiLoss
  loss_config_list:
    - CTCLoss:
    - NRTRLoss:

PostProcess:
  name: CTCLabelDecode

Metric:
  name: RecMetric
  main_indicator: acc
  ignore_space: false

Train:
  dataset:
    name: SimpleDataSet
    data_dir: /content/rec_train/
    label_file_list:
      - /content/rec_train/train_rec_label.txt
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - RecAug:
      - MultiLabelEncode:
          gtc_encode: NRTRLabelEncode
      - RecResizeImg:
          image_shape: [3, 48, 320]
      - KeepKeys:
          keep_keys: [image, label_ctc, label_gtc, length, valid_ratio]
  loader:
    shuffle: true
    batch_size_per_card: 256
    drop_last: true
    num_workers: 8

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: /content/rec_val/
    label_file_list:
      - /content/rec_val/val_rec_label.txt
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - MultiLabelEncode:
          gtc_encode: NRTRLabelEncode
      - RecResizeImg:
          image_shape: [3, 48, 320]
      - KeepKeys:
          keep_keys: [image, label_ctc, label_gtc, length, valid_ratio]
  loader:
    shuffle: false
    drop_last: false
    batch_size_per_card: 256
    num_workers: 8
"""

Path('/content/rec_finetune.yml').write_text(rec_yaml.strip(), encoding='utf-8')
print('rec_finetune.yml (v10: epoch 20 / lr 0.0001 / warmup 4 / freeze 미사용) 저장 완료')

## 11. Fine-tuning 실행

Rec 학습(epoch 20, lr 0.0001) 기준 A100에서 v4(epoch 5)가 약 30분이었던 것 대비 대략 4배, **2시간 안팎** 예상(데이터 양·detection 크롭 비율에 따라 변동). 중간에 정점을 찍고 내려갈 수 있으니 `eval_batch_step: [0, 150]`로 촘촘히 내부 eval을 찍어서 export 때 `best_accuracy`로 고른다. Det 학습은 이 버전에서 수행하지 않음(사전학습 PP-OCRv5_server_det 그대로 사용).

In [ ]:
# Section 11 실행 전 준비: rec_val 재생성 (Fix 18: crop_v7 + rotation_label_deg 원복, train과 동일)
import shutil, zipfile as _zf

# ── val raw crop 압축 해제 (crop_v7_val.zip) ──────────────────────────────────
VAL_CROP_ZIP    = DRIVE_BASE / 'crop_v7_val.zip'
VAL_CROP_DIR_RAW = Path('/content/crops_v7_val')
if VAL_CROP_ZIP.exists():
    if not VAL_CROP_DIR_RAW.exists() or not any(VAL_CROP_DIR_RAW.iterdir()):
        print('val crop 압축 해제 중...')
        with _zf.ZipFile(VAL_CROP_ZIP, 'r') as zf:
            zf.extractall(VAL_CROP_DIR_RAW)
    print(f'val raw crop: {sum(1 for _ in VAL_CROP_DIR_RAW.iterdir())}개')
else:
    VAL_CROP_DIR_RAW = CROP_DIR
    print('[WARN] crop_v7_val.zip 없음 → train crop 폴더 사용')

# ── val manifest (rotation_label_deg + bbox_w/h, crop_v7_manifest.csv에 이미 병합됨) 로드 ──
df_val_setup = pd.read_csv(MANIFEST_CSV, low_memory=False)
df_val_setup = df_val_setup[df_val_setup['split'] == 'val']
rot_map_val_tmp = df_val_setup.set_index('object_id')['rotation_label_deg'].to_dict()
bbox_lookup   = df_val_setup.set_index('object_id')[['bbox_w', 'bbox_h']].to_dict('index')

# ── val 정답: val_rec_label_v3.txt 로드 (rotation_best_text로 검증된 값, 마크/품질 필터 적용됨) ──
val_answer_lines = (DRIVE_BASE / 'val_rec_label_v3.txt').read_text(encoding='utf-8').strip().split('\n')
label_map = {}
for line in val_answer_lines:
    parts = line.split('\t')
    if len(parts) != 2:
        continue
    oid = Path(parts[0]).stem
    label_map[oid] = parts[1].strip()
print(f'val_rec_label_v3.txt 로드: {len(val_answer_lines)}줄 → object_id {len(label_map)}개')

# ── rec_val 이미지 재생성 (Fix 5: train과 동일하게 detection으로 텍스트 영역만 크롭) ──
VAL_REC_LOCAL = Path('/content/rec_val')
if VAL_REC_LOCAL.exists(): shutil.rmtree(str(VAL_REC_LOCAL))
(VAL_REC_LOCAL / 'crops').mkdir(parents=True)

new_labels, skip_count, skip_no_det_val, skip_multi_box_val, fallback_whole_crop_val = [], 0, 0, 0, 0
for oid, label_text in tqdm(label_map.items(), desc='rec_val 재생성'):
    src = VAL_CROP_DIR_RAW / f'{oid}.png'
    if not src.exists():
        skip_count += 1
        continue
    bb     = bbox_lookup.get(oid, {})
    bbox_w = float(bb.get('bbox_w', 0) or 0)
    bbox_h = float(bb.get('bbox_h', 0) or 0)
    img = prepare_img(str(src), bbox_w, bbox_h, rot_angle=rot_map_val_tmp.get(str(oid)))
    text_crop, n_boxes = crop_text_region(img)
    if text_crop is None:
        # Fix 14: detection 실패해도 라벨에서 빼지 않고 전체 crop으로 fallback —
        # 최종 eval(Section 12)과 같은 모집단을 유지해서 best_accuracy 선택이 '쉬운 부분집합'
        # 기준으로 왜곡되지 않도록 함 (코드 리뷰 지적).
        if n_boxes == 0:
            skip_no_det_val += 1
        else:
            skip_multi_box_val += 1
        text_crop = img
        fallback_whole_crop_val += 1
    text_crop = resolve_180_flip(text_crop)
    cv2.imwrite(str(VAL_REC_LOCAL / 'crops' / f'{oid}.png'), text_crop)
    new_labels.append(f'crops/{oid}.png\t{label_text}')

(VAL_REC_LOCAL / 'val_rec_label.txt').write_text('\n'.join(new_labels), encoding='utf-8')
print(f'rec_val 재생성 완료: {len(new_labels)}건  '
      f'(이미지 없음 스킵: {skip_count}건, detection 실패→전체 crop fallback: {fallback_whole_crop_val}건 '
      f'[no_det={skip_no_det_val}, multi_box={skip_multi_box_val}] — Fix 14: 더 이상 라벨에서 제외하지 않음)')

In [ ]:
!mkdir -p /content/pretrain_models
!wget -nc -q -P /content/pretrain_models https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_rec_pretrained.pdparams
!ls -lh /content/pretrain_models/

In [ ]:
# Fix 16 (코드 리뷰 반영): paddleocr==3.7.0(pip, 추론용)과 이 git clone(main, 학습 스크립트용)이
# 서로 다른 시점 조합일 수 있음 — main에 정확히 대응하는 태그를 안전하게 확정할 방법이 없어
# 우선 main을 그대로 쓰되, 실제로 어떤 커밋이 쓰였는지 항상 출력한다. 재현성이 필요하면
# 이 해시를 기록해뒀다가 다음 실행 때 `git checkout <hash>`로 고정할 것.
!git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git /content/PaddleOCR 2>/dev/null \
    || echo '이미 존재함'
%cd /content/PaddleOCR
!pip install -r requirements.txt -q
!echo "[PaddleOCR repo commit] $(git rev-parse HEAD)"

In [ ]:
# Recognition fine-tuning (~6시간)
# Fix 21: 런타임 연결이 끊기면 /content(휘발성)에 있던 체크포인트가 전부 사라져서
# 처음부터 다시 돌려야 하는 사고를 겪음 — 재발 방지로 학습을 백그라운드 프로세스로 돌리면서
# 주기적으로 Drive에 체크포인트를 백업한다.
import subprocess, time, shutil

BACKUP_INTERVAL_SEC = 600  # 10분마다 백업 (끊겨도 최대 10분치만 손실)
backup_dst = DRIVE_BASE / 'models' / 'rec_finetune_v10_inprogress'
ckpt_src = Path('/content/PaddleOCR/output/rec_finetune')

proc = subprocess.Popen(
    ['python', 'tools/train.py', '-c', '/content/rec_finetune.yml'],
    cwd='/content/PaddleOCR',
)

last_backup = 0.0
while proc.poll() is None:
    time.sleep(5)
    if time.time() - last_backup >= BACKUP_INTERVAL_SEC:
        if ckpt_src.exists():
            shutil.copytree(str(ckpt_src), str(backup_dst), dirs_exist_ok=True)
            print(f'[백업 {time.strftime("%H:%M:%S")}] 체크포인트를 Drive({backup_dst})에 백업 완료')
        last_backup = time.time()

print(f'학습 프로세스 종료, exit code: {proc.returncode}')
if ckpt_src.exists():
    shutil.copytree(str(ckpt_src), str(backup_dst), dirs_exist_ok=True)
    print('학습 종료 후 최종 체크포인트도 Drive에 백업 완료')

assert proc.returncode == 0, (
    f'tools/train.py가 비정상 종료(exit code {proc.returncode})했습니다. '
    f'Drive의 rec_finetune_v10_inprogress에 마지막 백업이 남아있으니, '
    f'Global.checkpoints=<복원한 경로>/latest 로 재개할 것.'
)

In [ ]:
import glob as _glob, re as _re

# Fix 9: val_rec_label_v3.txt 기반 내부 eval은 v4에서 실측으로 신뢰도가 확인됨(0.119→0.387
# 꾸준히 상승하는 정상적인 곡선). 20 epoch로 늘리면 중간에 정점을 찍고 내려갈 수 있어,
# 이번엔 best_accuracy(정점 epoch)를 우선 사용 — latest는 fallback으로만.
ckpt_base = Path('/content/PaddleOCR/output/rec_finetune')

if (ckpt_base / 'best_accuracy.pdparams').exists():
    ckpt_path = str(ckpt_base / 'best_accuracy')
    print('best_accuracy 체크포인트 사용 (내부 eval 정점 epoch)')
elif (ckpt_base / 'latest.pdparams').exists():
    ckpt_path = str(ckpt_base / 'latest')
    print('best_accuracy 없음 → latest(마지막 epoch) 체크포인트 사용')
else:
    candidates = _glob.glob(str(ckpt_base / 'iter_epoch_*.pdparams'))
    if not candidates:
        raise FileNotFoundError(f'체크포인트 없음: {ckpt_base}  (학습이 완료되었는지 확인)')
    # 파일명 문자열 정렬(iter_epoch_10 < iter_epoch_2)이 아니라 epoch 숫자 기준으로 정렬
    candidates.sort(key=lambda p: int(_re.search(r'iter_epoch_(\d+)', p).group(1)))
    ckpt_path = candidates[-1].replace('.pdparams', '')
    print(f'best_accuracy/latest 없음 → epoch 숫자 최댓값 fallback: {Path(ckpt_path).name}')

print(f'export 체크포인트: {ckpt_path}')
!python tools/export_model.py -c /content/rec_finetune.yml -o Global.pretrained_model={ckpt_path} Global.save_inference_dir=/content/rec_inference/
!ls /content/rec_inference/

In [ ]:
# ── Drive 저장 ────────────────────────────────────────────────────────────────
import shutil
from pathlib import Path

(DRIVE_BASE / 'models').mkdir(parents=True, exist_ok=True)

# 학습 체크포인트 전체
shutil.copytree('/content/PaddleOCR/output/rec_finetune',
                str(DRIVE_BASE / 'models/rec_finetune_v10'), dirs_exist_ok=True)

# inference 모델
shutil.copytree('/content/rec_inference',
                str(DRIVE_BASE / 'models/rec_inference_v10'), dirs_exist_ok=True)

# Fix 1: train_rec_label_v3.txt(검증 완료된 원본 정답 매핑)는 절대 덮어쓰지 않음.
# 증강본 라벨은 별도 파일로만 백업 (참고용, object_id 기반 매핑이 아니므로 검증에 쓰지 말 것)
shutil.copy('/content/rec_train/train_rec_label.txt',
            str(DRIVE_BASE / 'train_rec_label_augmented_v10.txt'))

print('Drive 저장 완료')

## 12. Fine-tuned 모델 평가 (val split)

**주의(v10, Fix 24)**: val split이 이제 Section 9-3에서 학습 데이터에도 포함된다. 따라서 이 Section의 평가는 더 이상 순수한 held-out 성능이 아니라 leakage가 섞인 값이다 — 팀 결정으로 감수. v9까지의 "honest" 숫자와 직접 비교하지 말 것.

- 멀티앵글 없음 (단일 추론)
- 저대비 adaptive config 없음
- val split 전체 평가

> **사전 조건**: Section 11 export 셀 완료 후 실행

In [ ]:
# ── 12-1. 회전각 confidence 탐색 함수 (Fix 19, Fix 23) ────────────────────────
# Fix 19: Section 12(eval)에서는 rotation_label_deg(정답 텍스트와 대조해서 만든 leakage 라벨)를
# 쓰지 않는다 — 실제 배포에서는 정답을 모르니 이 라벨 자체가 존재할 수 없어서, eval 성능이
# 실제 배포 성능보다 부풀려짐. YOLO OBB 각도도 대안으로 검토했지만 신뢰 가능한 각도가 전체의
# 7%뿐이라 커버리지가 너무 낮음. 대신 OCR 자신의 confidence를 각도 선택 기준으로 쓴다 —
# conf-CER 상관 -0.734로 확인됨(confidence 높을수록 실제로 잘 읽었을 가능성이 높음). 정답도
# YOLO도 필요 없이 이미지 한 장만으로 계산 가능해서 실제 배포에서 그대로 재현 가능하다.
# Section 9(train)는 그대로 rotation_label_deg를 씀 — 정답을 참고해 좋은 학습 페어를 만드는
# 것 자체는 leakage가 아니라 정상적인 데이터 큐레이션이라 문제없음.
# Fix 23 (Fix 22 수정): ablation_v2에서 크롭 없이 rotate_only(poly 미세 회전보정) 단독을
# 적용해도 baseline(아무 후처리 없음)보다 더 나쁘게 나옴(EM -0.056) — 크롭을 빼도 이 미세보정
# 자체가 여전히 독이 된다는 뜻(해당 arm은 resolve_180_flip을 호출하지 않았고, 180도 처리는
# 세 arm 모두 동일하게 use_textline_orientation=True 내장 처리에만 의존했음 — resolve_180_flip
# 자체는 이 실험에서 나쁘다고 확인된 적 없고, 오히려 3-way/4-way ablation에서 크롭 위에
# 얹었을 때는 근소하게 긍정적이었음+0.001~+0.003). 반면 multi-angle(30도 단위 굵은 회전
# 후보 + confidence 선택) 자체는 적용 여부 비교에서 도움이 되는 걸로 확인됨. 그래서 이 함수는
# rotate_only 호출을 제거하고, resolve_180_flip은 (근거 없이 비용만 늘리는 걸 피하기 위해)
# 마찬가지로 넣지 않은 채, 30도 단위로 통째로 돌린 이미지를 그대로 OCR에 넣어 confidence로
# 최적 각도만 고른다.

def resolve_rotation_by_confidence(img, ocr, angle_step=30):
    """0~330도를 angle_step 간격(기본 12번)으로 통째로 돌려보고(rotate_image), 그 결과를
    그대로 OCR에 넣어 recognition confidence가 가장 높은 각도를 채택한다. poly 기반 미세
    회전보정(rotate_only)이나 180도 보정(resolve_180_flip)은 추가로 적용하지 않는다 — ablation에서
    이 추가 보정들이 crop 없이 적용해도 baseline보다 나쁘게 나와 순수 multi-angle만 남김(Fix 23).
    반환: (선택된 이미지 또는 None, 그 각도의 OCR page 결과 또는 None, best_conf)"""
    best_conf, best_img, best_page = -1.0, None, None
    for angle in range(0, 360, angle_step):
        rimg = rotate_image(img, angle) if angle else img
        ocr_result = ocr.predict(rimg, text_det_thresh=0.3)
        if not ocr_result:
            continue
        _, ocr_text, ocr_conf, polys, confs = _extract_ocr(ocr_result[0])
        if not np.isnan(ocr_conf) and ocr_conf > best_conf:
            best_conf, best_img, best_page = ocr_conf, rimg, ocr_result[0]
    return best_img, best_page, best_conf


# ── 12-2. inference 함수 ──────────────────────────────────────────────────────
# Fix 13/23: train(Section 9)은 crop_text_region으로 미리 잘라야 하지만(recognition 전용
# 모델이라 내장 detection 없음), eval은 PaddleOCR 풀 파이프라인이 자체 detection을 다시
# 수행하므로 crop_text_region도 rotate_only도 쓰지 않고 30도 단위 multi-angle 회전만 적용한다.
# Fix 14: detection 실패를 강제 0점 처리하지 않고 전체 crop으로 fallback해서 실제로
# recognition을 시도한 뒤 score_one/exact_match로 정상 채점한다.
# prepare_img → Section 5에서 정의

def run_inference_val(row, ocr):
    result = {
        'object_id': str(row['object_id']),
        'crop_path': row['crop_path'],
        'target_text': row.get('target_text', ''),
        'ocr_text': '', 'ocr_conf': float('nan'), 'score': float('nan'),
        'rec_polys': [], 'rec_confs': [], 'error': '',
    }
    try:
        bbox_w = float(row.get('bbox_w', 0) or 0)
        bbox_h = float(row.get('bbox_h', 0) or 0)
        # Fix 19: rot_angle=None — 라벨 기반 1차 보정 없이 preprocess+upscale만 하고,
        # 회전 자체는 아래 confidence 탐색이 전담한다.
        base_img = prepare_img(row['crop_path'], bbox_w, bbox_h, rot_angle=None)
        text_crop, best_page, best_conf = resolve_rotation_by_confidence(base_img, ocr, angle_step=30)
        if best_page is None:
            # Fix 14: 12개 각도 전부 실패해도 전체 crop으로 마지막 시도(완전 포기 안 함)
            ocr_result = ocr.predict(base_img, text_det_thresh=0.3)
            if not ocr_result:
                result['error'] = 'multi-angle 탐색 + 전체 crop 시도 모두 실패'
                return result
            best_page = ocr_result[0]
            result['error'] = 'multi-angle 탐색 실패 → 전체 crop으로 fallback'
        _, ocr_text, ocr_conf, polys, confs = _extract_ocr(best_page)
        result.update({'ocr_text': ocr_text, 'ocr_conf': ocr_conf,
                       'rec_polys': [p.tolist() if hasattr(p,'tolist') else p for p in polys],
                       'rec_confs': confs})
        cands = row.get('target_candidates', [])
        if isinstance(cands, list) and cands:
            s = score_one(ocr_text, cands)
            if s is not None: result['score'] = s
    except Exception as exc:
        result['error'] = str(exc)
    return result

In [ ]:
# ── 12-3. OCR 초기화 (Fix 18: crop_v7 원복 후에도 textline orientation ON 유지) ──────
# v7 원래는 False였음(YOLO/OBB 없이 rot_map이 방향까지 다 잡아준다고 가정). 하지만 rot_map은
# 추정값이라 방향 판단이 틀릴 수 있고, run_inference_val에서 이미 crop_text_region+
# resolve_180_flip을 거친 뒤라 여기서 True로 둬도 대부분 '이미 맞음'을 재확인만 하는
# 무해한 안전망이라 True 유지.

import subprocess
from paddleocr import PaddleOCR

# PaddleOCR 클론
if not Path('/content/PaddleOCR').exists():
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/PaddlePaddle/PaddleOCR.git',
                    '/content/PaddleOCR'], check=True)

# ── Fix 15 (코드 리뷰 반영): stale 모델을 몰래 평가하는 사고 방지 ──────────────────
# 이전엔 Drive의 rec_inference_v9가 없으면 조용히 rec_inference_v8(구버전)로 폴백했음 —
# 방금 학습한 모델이 아니라 예전 버전을 평가하고도 눈치채기 어려운 구조였음.
# 1) 같은 세션에서 방금 export한 로컬 모델이 이미 있으면 Drive 왕복 없이 그대로 사용.
# 2) 로컬에 없으면(새 세션 등) Drive에서 복원하되, v9가 없으면 예전 버전으로 폴백하지 않고
#    명시적으로 멈춘다 — 잘못된 버전을 평가하는 것보다 여기서 막히는 게 훨씬 안전하다.
rec_inf_dst = Path('/content/rec_inference')
if rec_inf_dst.exists() and any(rec_inf_dst.iterdir()):
    print(f'[INFO] 로컬에 이미 존재하는 {rec_inf_dst}를 그대로 사용 '
          f'(같은 세션에서 방금 export한 모델로 추정 — Drive 왕복 생략).')
else:
    rec_inf_src = DRIVE_BASE / 'models' / 'rec_inference_v10'
    assert rec_inf_src.exists(), (
        f'{rec_inf_src} 없음 — Section 10(export)과 Drive 저장 셀을 먼저 실행했는지 확인할 것. '
        f'예전 버전(v9 등)으로 자동 폴백하지 않음(잘못된 모델을 평가하는 사고 방지). '
        f'예전 버전을 일부러 재평가하려면 이 셀의 경로를 직접 바꿀 것.'
    )
    print(f'복원 소스: {rec_inf_src.name}')
    shutil.copytree(str(rec_inf_src), str(rec_inf_dst))
    print('rec_inference 복원 완료')

# fine-tuned 모델 초기화
_ocr_ft = PaddleOCR(
    device='gpu',
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=True,
    text_detection_model_name='PP-OCRv5_server_det',
    text_recognition_model_name='PP-OCRv5_server_rec',  # 명시 필수(리뷰 지적) - 다만 실측상
                                                          # 결과에는 영향 없었음(아래 참고)
    text_recognition_model_dir='/content/rec_inference',
)
print('fine-tuned 초기화 완료')
# 참고: model_name 유무로 실제 로드된 가중치가 바뀌는지 이미 테스트함 -
# 결과가 동일했고(0.033/0.006), recognition-only 테스트에서 커스텀 정답 어휘로 train 91.4%를
# 맞힌 것도 확인해서, model_dir만으로도 올바른 파인튜닝 가중치가 로드됐다는 근거가 있음.
# 그래도 이름 명시는 위생상 유지.

In [ ]:
# ── 12-4. val 데이터 로드 ─────────────────────────────────────────────────────
df_val_mf = pd.read_csv(MANIFEST_CSV, low_memory=False)
df_val_mf = df_val_mf[df_val_mf['split'] == 'val'].reset_index(drop=True)
print(f'val manifest: {len(df_val_mf)}건')

VAL_CROP_ZIP = DRIVE_BASE / 'crop_v7_val.zip'  # Fix 18: crop_v7로 원복
VAL_CROP_DIR = Path('/content/crops_v7_val')
if VAL_CROP_ZIP.exists():
    if not VAL_CROP_DIR.exists() or not any(VAL_CROP_DIR.iterdir()):
        print('val crop 압축 해제 중...')
        with zipfile.ZipFile(VAL_CROP_ZIP, 'r') as zf:
            zf.extractall(VAL_CROP_DIR)
    val_crop_dir = VAL_CROP_DIR
else:
    val_crop_dir = CROP_DIR
    print('[INFO] crop_v7_val.zip 없음 → CROP_DIR 사용')
print(f'val crop dir: {val_crop_dir}')

df_val_mf['crop_path'] = df_val_mf['object_id'].apply(
    lambda oid: str(val_crop_dir / f'{oid}.png'))
tgt = df_val_mf.apply(build_target_for_row, axis=1, result_type='expand')
df_val_mf = pd.concat([df_val_mf, tgt], axis=1)
df_val_mf['_exists'] = df_val_mf['crop_path'].apply(lambda p: Path(p).exists())
df_val = df_val_mf[df_val_mf['_exists']].reset_index(drop=True)
print(f'val 사용 가능: {len(df_val)}건')

In [ ]:
# ── 12-5. 평가 실행 ───────────────────────────────────────────────────────────
val_records = []
for _, row in tqdm(df_val.iterrows(), total=len(df_val), desc='평가 (fine-tuned)'):
    val_records.append(run_inference_val(row, _ocr_ft))

df_val_result = pd.DataFrame(val_records)
valid = df_val_result['score'].notna()
cand_map = df_val.set_index('object_id')['target_candidates'].to_dict()
df_val_result['exact_match'] = df_val_result.apply(
    lambda r: exact_match(r['ocr_text'], cand_map.get(r['object_id'], [])), axis=1)

print('=' * 40)
print(f'[주] 글자 정확도  ({valid.sum()}건) : {df_val_result.loc[valid, "score"].mean():.3f}')
print(f'[보] Exact Match ({len(df_val_result)}건) : {df_val_result["exact_match"].mean():.3f}  '
      f'(Fix 10: substring 관대 매칭 제거, 정확히 일치해야 함)')
print('=' * 40)
df_val_result.to_csv(RESULT_DIR / 'ft_v10_val_result.csv', index=False)
print('결과 저장 완료')

### 12-5-1. pretrained 기준선과 비교 (Fix 5 검증)

fine-tuned가 pretrained(파인튜닝 전)보다 나아졌는지 같은 val set으로 직접 비교. v3에서는 pretrained가 글자정확도 0.811 / EM 0.726인데 fine-tuned는 0.033 / 0.006으로 훨씬 나빴음 — 이번 v4가 이 기준선을 넘는지가 성공 여부의 핵심 판단 기준.

In [ ]:
# ── 12-5-1. pretrained 기준선과 비교 ─────────────────────────────────────────
_ocr_pretrained = PaddleOCR(
    device='gpu',
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=True,
    text_detection_model_name='PP-OCRv5_server_det',
    text_recognition_model_name='PP-OCRv5_server_rec',  # pretrained (파인튜닝 안 함)
)

pretrained_records = []
for _, row in tqdm(df_val.iterrows(), total=len(df_val), desc='평가 (pretrained 기준선)'):
    pretrained_records.append(run_inference_val(row, _ocr_pretrained))

df_pretrained_result = pd.DataFrame(pretrained_records)
valid_pre = df_pretrained_result['score'].notna()
df_pretrained_result['exact_match'] = df_pretrained_result.apply(
    lambda r: exact_match(r['ocr_text'], cand_map.get(r['object_id'], [])), axis=1)

print('=' * 50)
print(f'[v10 fine-tuned] 글자 정확도: {df_val_result.loc[valid, "score"].mean():.3f}  '
      f'Exact Match: {df_val_result["exact_match"].mean():.3f}')
print(f'[pretrained]    글자 정확도: {df_pretrained_result.loc[valid_pre, "score"].mean():.3f}  '
      f'Exact Match: {df_pretrained_result["exact_match"].mean():.3f}')

# 이전 버전 결과가 저장돼 있으면 같이 비교 (v7: crop_v7 기반 마지막 버전, v8/v9는 OBB straighten crop을 시도했다가 Fix 18로 crop_v7로 다시 원복함)
# Fix 10: 저장된 exact_match 컬럼은 옛 substring 관대 매칭 기준이라 그대로 못 믿음 -> ocr_text로 재계산
for _tag in ['ft_v9_val_result.csv', 'ft_v8_val_result.csv', 'ft_v7_val_result.csv', 'ft_v6_val_result.csv', 'ft_v5_val_result.csv', 'ft_v4_val_result.csv']:
    _csv = RESULT_DIR / _tag
    if _csv.exists():
        _df_prev = pd.read_csv(_csv)
        _valid_prev = _df_prev['score'].notna()
        _em_prev = _df_prev.apply(
            lambda r: exact_match(r['ocr_text'], cand_map.get(r['object_id'], [])), axis=1)
        print(f'[{_tag}] 글자 정확도: {_df_prev.loc[_valid_prev, "score"].mean():.3f}  '
              f'Exact Match(재계산): {_em_prev.mean():.3f}')
print('=' * 50)

if df_val_result['exact_match'].mean() <= df_pretrained_result['exact_match'].mean():
    print('경고: v10 fine-tuned가 pretrained 기준선을 못 넘었습니다. 학습 방식을 재검토해야 합니다.')
else:
    print('v10 fine-tuned가 pretrained 기준선을 넘었습니다. (주: val이 학습에 포함돼 leakage 영향 있음, Fix 24)')

df_pretrained_result.to_csv(RESULT_DIR / 'pretrained_baseline_val_result.csv', index=False)

### 12-5-2. confidence 탐색 vs simple blind 비교

`resolve_rotation_by_confidence`(Fix 19/20)의 12각도 탐색이 실제로 얼마나 기여하는지 확인하는 ablation. **simple blind**는 회전 탐색 없이(각도 0, 즉 crop_v7 원본 그대로) `crop_text_region`(Fix 12)+`resolve_180_flip`(Fix 11)만 한 번 적용한 결과 — 나머지 파이프라인은 동일하고 '여러 각도를 시도하는지 여부'만 다르다. 이 차이가 작으면 multi-angle 탐색의 비용 대비 효과가 낮다는 뜻이고, 크면 탐색이 실질적으로 기여하고 있다는 뜻이다.

In [ ]:
def run_inference_val_blind(row, ocr):
    """12-5-2: multi-angle confidence 탐색 없이, 회전 시도 0번(원본 그대로)으로 recognition만
    수행. resolve_rotation_by_confidence(Fix 19/20)가 실제로 얼마나 기여하는지 확인하기 위한
    ablation 전용 함수 — run_inference_val과 나머지 로직은 동일, 각도 탐색만 뺌."""
    result = {
        'object_id': str(row['object_id']),
        'crop_path': row['crop_path'],
        'target_text': row.get('target_text', ''),
        'ocr_text': '', 'ocr_conf': float('nan'), 'score': float('nan'),
        'rec_polys': [], 'rec_confs': [], 'error': '',
    }
    try:
        bbox_w = float(row.get('bbox_w', 0) or 0)
        bbox_h = float(row.get('bbox_h', 0) or 0)
        base_img = prepare_img(row['crop_path'], bbox_w, bbox_h, rot_angle=None)
        text_crop, n_boxes = crop_text_region(base_img)
        if text_crop is None:
            text_crop = base_img  # Fix 14와 동일: 실패해도 전체 crop으로 fallback
        text_crop = resolve_180_flip(text_crop)
        ocr_result = ocr.predict(text_crop, text_det_thresh=0.3)
        if not ocr_result: return result
        _, ocr_text, ocr_conf, polys, confs = _extract_ocr(ocr_result[0])
        result.update({'ocr_text': ocr_text, 'ocr_conf': ocr_conf,
                       'rec_polys': [p.tolist() if hasattr(p,'tolist') else p for p in polys],
                       'rec_confs': confs})
        cands = row.get('target_candidates', [])
        if isinstance(cands, list) and cands:
            s = score_one(ocr_text, cands)
            if s is not None: result['score'] = s
    except Exception as exc:
        result['error'] = str(exc)
    return result


blind_records = []
for _, row in tqdm(df_val.iterrows(), total=len(df_val), desc='평가 (simple blind, 회전탐색 없음)'):
    blind_records.append(run_inference_val_blind(row, _ocr_ft))

df_blind_result = pd.DataFrame(blind_records)
valid_blind = df_blind_result['score'].notna()
df_blind_result['exact_match'] = df_blind_result.apply(
    lambda r: exact_match(r['ocr_text'], cand_map.get(r['object_id'], [])), axis=1)

print('=' * 50)
print(f'[confidence search (Fix 19/20), 12각도] 글자 정확도: '
      f'{df_val_result.loc[valid, "score"].mean():.3f}  '
      f'Exact Match: {df_val_result["exact_match"].mean():.3f}')
print(f'[simple blind (회전탐색 없음, 1각도)]    글자 정확도: '
      f'{df_blind_result.loc[valid_blind, "score"].mean():.3f}  '
      f'Exact Match: {df_blind_result["exact_match"].mean():.3f}')
print('=' * 50)
df_blind_result.to_csv(RESULT_DIR / 'ft_v10_blind_val_result.csv', index=False)
print('결과 저장 완료')

In [ ]:
# ── 12-6. 원인 진단 ───────────────────────────────────────────────────────────
# OCR이 무엇을 읽고 있는지 확인

empty_count = (df_val_result['ocr_text'] == '').sum()
error_count = (df_val_result['error'] != '').sum()
print(f'OCR 빈 결과 (미검출): {empty_count}건 ({empty_count/len(df_val_result)*100:.1f}%)')
print(f'에러 발생: {error_count}건')
print(f'score 분포:\n{df_val_result["score"].describe().round(3)}')

print('\n── 샘플: score 높은 순 10건 ──')
display(df_val_result[['object_id','target_text','ocr_text','score','exact_match']]
        .dropna(subset=['score']).sort_values('score', ascending=False).head(10))

print('\n── 샘플: score 낮은 순 10건 ──')
display(df_val_result[['object_id','target_text','ocr_text','score','exact_match']]
        .dropna(subset=['score']).sort_values('score').head(10))